In [ ]:
# @title 🛠️ Step 0: Install Dependencies
# ==============================================================================
# Run once per Colab session. Two things worth knowing:
#
#  * Docking uses the `vina` EXECUTABLE installed by apt, not the `vina` Python
#    package. That package publishes no wheel for Python 3.13 -- which is what
#    Colab now runs -- and building it from source there fails. The executable is
#    the same engine (v1.2.5) and gives identical affinities.
#
#  * Each pip package is installed separately. In a single `pip install`, one
#    failing build aborts the entire command: a vina build failure silently took
#    RDKit and Biopython down with it, and nothing said so until a later import.
# ==============================================================================
print("Setting up the drug discovery environment...")

# 1. System binaries (OpenBabel for file conversion, AutoDock Vina for docking)
!apt-get update -y -qq
!apt-get install -y -qq openbabel autodock-vina

# 2. Python packages, one at a time so a single failure cannot cascade
import importlib.util
import shutil
import subprocess
import sys

PACKAGES = [
    "rdkit>=2023.3.1", "chembl_webresource_client", "biopython",
    "pandas>=1.5", "numpy>=1.23", "scikit-learn>=1.2", "scipy>=1.9",
    "matplotlib", "seaborn", "joblib", "requests", "py3Dmol",
]

failed = []
for spec in PACKAGES:
    result = subprocess.run([sys.executable, "-m", "pip", "install", "-q", spec],
                            capture_output=True, text=True)
    print(f"  {spec:<32} {'ok' if result.returncode == 0 else 'FAILED'}")
    if result.returncode != 0:
        failed.append(spec)
        tail = (result.stderr or result.stdout or "").strip().splitlines()
        if tail:
            print(f"      {tail[-1][:160]}")

# 3. Verify what the pipeline actually needs
missing = [m for m in ("rdkit", "sklearn", "pandas", "Bio", "scipy", "matplotlib")
           if importlib.util.find_spec(m) is None]
vina_bin = shutil.which("vina")

print()
print(f"  Python           : {sys.version.split()[0]}")
print(f"  Vina executable  : {vina_bin or 'NOT FOUND'}")
if vina_bin:
    subprocess.run([vina_bin, "--version"])

if missing or not vina_bin:
    raise SystemExit(
        f"Setup incomplete.\n"
        f"  missing Python modules : {missing or 'none'}\n"
        f"  Vina executable        : {'found' if vina_bin else 'NOT FOUND'}\n"
        f"Re-run this cell. If Vina is still missing, run:\n"
        f"  !apt-get install -y autodock-vina")

print("\n✅ Environment ready. Now run Step 0b (configuration).")


In [ ]:
# @title ⚙️ Step 0b: Pipeline Configuration — THE ONLY CELL YOU EDIT TO SWITCH TARGET
# ==============================================================================
# Every published run (KRAS, EGFR, mTOR) executes *identical* code. The only
# difference between them is which entry of TARGETS is selected below.
#
# To switch target : change TARGET on the marked line.
# To add a target  : add an entry to TARGETS in pipeline_common.py.
# Never edit a later cell to change a target — later cells read this config.
# ==============================================================================
import importlib
import os
import sys
import urllib.request
from pathlib import Path

BASE_DIR = Path(os.environ.get("PIPELINE_BASE_DIR", "/content"))
BASE_DIR.mkdir(parents=True, exist_ok=True)
if str(BASE_DIR) not in sys.path:
    sys.path.insert(0, str(BASE_DIR))

# ------------------------------------------------------------------------------
# Locate the shared module. It lives beside the notebook in the repository; on a
# fresh Colab session there is nothing beside the notebook, so fetch it once.
# ------------------------------------------------------------------------------
REPO_RAW = ("https://raw.githubusercontent.com/GeeYuXuan/"
            "Integrated_Computational_Drug_Discovery_Pipeline_Generalized_for_"
            "Small_Molecule_Drugs/main")
_module = BASE_DIR / "pipeline_common.py"

if not _module.exists() and not Path("pipeline_common.py").exists():
    try:
        print("Fetching pipeline_common.py ...")
        urllib.request.urlretrieve(f"{REPO_RAW}/pipeline_common.py", _module)
        print(f"  ✓ saved to {_module}")
    except Exception as exc:
        raise SystemExit(
            f"Could not download pipeline_common.py ({exc}).\n"
            f"This file holds the shared configuration and helpers for every step.\n"
            f"Fix: upload pipeline_common.py from the repository into {BASE_DIR}, "
            f"then re-run this cell."
        )

import pipeline_common
importlib.reload(pipeline_common)   # pick up edits without restarting the kernel
from pipeline_common import *       # noqa: F401,F403  (helpers used by later steps)

# ==============================================================================
# >>>>>>>>>>  CHANGE THIS LINE ONLY:  "kras" | "egfr" | "mtor"  <<<<<<<<<<
# ==============================================================================
TARGET = "kras"
# ==============================================================================

CFG = pipeline_common.get_config(TARGET, base_dir=str(BASE_DIR))
pipeline_common.export_globals(CFG, globals())

print("\nConfiguration loaded. Every later cell reads it from pipeline_config.json,")
print("so you can restart the kernel and re-run any single step on its own.")


In [ ]:
#!/usr/bin/env python3
"""
Step 1: Target-agnostic inhibitor data acquisition and curation from ChEMBL.

Produces TWO datasets, deliberately:

  {target}_bioactivity_full.csv   every curated compound, full measured potency
                                  range. This is what the ML model in Step 6
                                  trains on — a model trained only on potent
                                  compounds never learns what makes a compound
                                  weak, and so cannot rank anything.

  {target}_inhibitors_cleaned.csv the potent subset (IC50 < POTENCY_CUTOFF_NM).
                                  This is what scaffold analysis and seed
                                  selection use, where "what do good compounds
                                  look like" is the actual question.

Target configuration comes from pipeline_common.py. Nothing here is specific to
KRAS, EGFR or mTOR.
"""

import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from chembl_webresource_client.new_client import new_client
from rdkit import Chem
import warnings
warnings.filterwarnings('ignore')

import pipeline_common
CFG = pipeline_common.get_config(quiet=True)
pipeline_common.export_globals(CFG, globals())

plt.switch_backend('Agg')

print("=" * 80)
print(f"STEP 1: DATA ACQUISITION — {TARGET_SEARCH_TERM} "
      f"({MUTANT_FILTER or 'wild type'})")
print("=" * 80)

# ==============================================================================
# 1. Target identification
# ==============================================================================
print("\n[1/6] Target identification")
print("-" * 80)

target_api = new_client.target
selected_target_id = None

if TARGET_CHEMBL_ID:
    try:
        target_data = target_api.get(TARGET_CHEMBL_ID)
        if target_data:
            print(f"  Direct match: {TARGET_CHEMBL_ID}")
            print(f"    Name    : {target_data.get('pref_name', 'N/A')}")
            print(f"    Organism: {target_data.get('organism', 'N/A')}")
            print(f"    Type    : {target_data.get('target_type', 'N/A')}")
            selected_target_id = TARGET_CHEMBL_ID
    except Exception as exc:
        print(f"  Direct lookup of {TARGET_CHEMBL_ID} failed ({exc})")

if selected_target_id is None:
    print(f"  Searching ChEMBL for '{TARGET_SEARCH_TERM}' (Homo sapiens, single protein)...")
    candidates = []
    for hit in target_api.search(TARGET_SEARCH_TERM):
        pref = (hit.get('pref_name') or '').upper()
        if (hit.get('organism') == 'Homo sapiens'
                and TARGET_SEARCH_TERM.upper() in pref
                and '/' not in pref):
            candidates.append(hit)
            print(f"    candidate: {hit['target_chembl_id']} — {hit['pref_name']} "
                  f"({hit.get('target_type')})")
    single = [c for c in candidates if c.get('target_type') == 'SINGLE PROTEIN']
    chosen = (single or candidates)
    if not chosen:
        raise RuntimeError(
            f"No human target matching '{TARGET_SEARCH_TERM}' in ChEMBL. Set "
            f"'chembl_id' explicitly for target '{CFG['target_name']}' in "
            f"pipeline_common.py.")
    selected_target_id = chosen[0]['target_chembl_id']
    print(f"  Selected: {selected_target_id} — {chosen[0]['pref_name']}")

print(f"  Using target: {selected_target_id}")

# ==============================================================================
# 2. Bioactivity retrieval
# ==============================================================================
print("\n[2/6] Bioactivity retrieval (IC50)")
print("-" * 80)

activities = new_client.activity.filter(
    target_chembl_id=selected_target_id, standard_type="IC50")

records = []
for i, act in enumerate(activities):
    if i and i % 1000 == 0:
        print(f"  retrieved {i} ...")
    records.append(act)

print(f"  Total activity records: {len(records)}")
if not records:
    raise RuntimeError(f"No IC50 data for {selected_target_id}.")

df = pd.DataFrame.from_records(records)

# ==============================================================================
# 3. Cleaning: units, relations, assay type, mutant
# ==============================================================================
print("\n[3/6] Cleaning")
print("-" * 80)


def to_nanomolar(value, unit):
    """Convert a potency to nM. Unknown units return NaN and are dropped —
    never silently assumed to already be nM."""
    factors = {'nM': 1.0, 'uM': 1e3, 'µM': 1e3, 'mM': 1e6, 'M': 1e9, 'pM': 1e-3}
    if pd.isna(value) or pd.isna(unit):
        return np.nan
    return value * factors[unit] if unit in factors else np.nan


n0 = len(df)
df = df.dropna(subset=['standard_value', 'standard_units'])
df['standard_value'] = pd.to_numeric(df['standard_value'], errors='coerce')
df = df[df['standard_value'] > 0]
print(f"  usable numeric values : {n0} -> {len(df)}")

if 'assay_type' in df.columns:
    n = len(df)
    df = df[df['assay_type'].isin(['B', 'F'])]
    print(f"  binding/functional    : {n} -> {len(df)}")

if 'standard_relation' in df.columns:
    n = len(df)
    df = df[(df['standard_relation'] == '=') | (df['standard_relation'].isna())]
    print(f"  exact measurements    : {n} -> {len(df)}")

print(f"  unit distribution     : {dict(df['standard_units'].value_counts())}")
df['ic50_nm'] = [to_nanomolar(v, u)
                 for v, u in zip(df['standard_value'], df['standard_units'])]
n = len(df)
df = df[df['ic50_nm'].notna() & (df['ic50_nm'] > 0)]
print(f"  convertible to nM     : {n} -> {len(df)}")

if MUTANT_FILTER:
    n = len(df)
    text = df.get('assay_description', pd.Series([''] * len(df), index=df.index))
    df = df[text.str.contains(MUTANT_FILTER, case=False, na=False)]
    print(f"  mentions '{MUTANT_FILTER}'  : {n} -> {len(df)}")
    print(f"    NOTE: this is a text match on the assay description, so an assay "
          f"comparing\n          {MUTANT_FILTER} against another variant also "
          f"matches. Check the assay list below.")
    if len(df) == 0:
        raise RuntimeError(
            f"No assay descriptions mention '{MUTANT_FILTER}'. Set "
            f"'mutant_filter' to None in pipeline_common.py to use all variants.")

# Where does the data actually come from? A set dominated by one assay describes
# one paper's compound series, not the target's chemistry.
if 'assay_chembl_id' in df.columns:
    counts = df['assay_chembl_id'].value_counts()
    print(f"\n  Provenance: {len(df)} measurements from {len(counts)} assays")
    for assay_id, c in counts.head(5).items():
        print(f"    {assay_id}: {c} ({100 * c / len(df):.0f}%)")
    if len(counts) and counts.iloc[0] / len(df) > 0.5:
        print(f"  !! {100 * counts.iloc[0] / len(df):.0f}% of the data comes from a "
              f"single assay. Scaffold and SAR conclusions\n     will largely "
              f"describe that one series — say so when reporting them.")

# ==============================================================================
# 4. Chemical standardisation
# ==============================================================================
print("\n[4/6] Chemical standardisation")
print("-" * 80)

smiles_col = next((c for c in ('canonical_smiles', 'smiles') if c in df.columns), None)
if smiles_col is None and 'molecule_structures' in df.columns:
    df['canonical_smiles'] = df['molecule_structures'].apply(
        lambda x: x.get('canonical_smiles') if isinstance(x, dict) else None)
    smiles_col = 'canonical_smiles'
if smiles_col is None:
    raise RuntimeError(f"No SMILES column in ChEMBL response. Columns: {list(df.columns)}")

df = df[df[smiles_col].notna()]

rows, n_bad, n_peptide = [], 0, 0
for i, (_, row) in enumerate(df.iterrows()):
    if i and i % 500 == 0:
        print(f"  standardised {i}/{len(df)} ...")
    try:
        mol = Chem.MolFromSmiles(row[smiles_col])
        if mol is None:
            n_bad += 1
            continue
        # Strip salts/counterions: keep the largest covalent fragment.
        frags = Chem.GetMolFrags(mol, asMols=True, sanitizeFrags=True)
        if len(frags) > 1:
            mol = max(frags, key=lambda m: m.GetNumHeavyAtoms())
        # This is a small-molecule pipeline: peptides and large macrocycles would
        # otherwise become "privileged scaffolds" and seeds for the generator.
        if CFG['exclude_peptides'] and pipeline_common.is_peptide_like(mol, CFG):
            n_peptide += 1
            continue
        rows.append({
            'molecule_chembl_id': row.get('molecule_chembl_id', 'N/A'),
            'canonical_smiles': Chem.MolToSmiles(mol, canonical=True),
            'ic50_nm': float(row['ic50_nm']),
            'assay_chembl_id': row.get('assay_chembl_id', 'N/A'),
        })
    except Exception:
        n_bad += 1

print(f"  standardised : {len(rows)}")
print(f"  unparseable  : {n_bad}")
if CFG['exclude_peptides']:
    print(f"  peptide-like removed: {n_peptide} "
          f"(>{CFG['peptide_max_amide_bonds']} amide bonds or a ring "
          f">{CFG['peptide_max_ring_size']} atoms)")
if not rows:
    raise RuntimeError("Nothing survived standardisation.")

std = pd.DataFrame(rows)

# ==============================================================================
# 5. Aggregate repeat measurements, then split full range vs potent hits
# ==============================================================================
print("\n[5/6] Aggregating repeat measurements")
print("-" * 80)

policy = CFG['duplicate_policy']
grouped = std.groupby('canonical_smiles')
agg = grouped.agg(
    ic50_nm=('ic50_nm', 'median' if policy == 'median' else 'min'),
    n_measurements=('ic50_nm', 'size'),
    ic50_min_nm=('ic50_nm', 'min'),
    ic50_max_nm=('ic50_nm', 'max'),
    molecule_chembl_id=('molecule_chembl_id', 'first'),
    assay_chembl_id=('assay_chembl_id', 'first'),
).reset_index()

print(f"  {len(std)} measurements -> {len(agg)} unique compounds "
      f"(policy: {policy} across assays)")
repeats = agg[agg['n_measurements'] > 1]
if len(repeats):
    spread = (np.log10(repeats['ic50_max_nm']) - np.log10(repeats['ic50_min_nm']))
    print(f"  {len(repeats)} compounds measured more than once; "
          f"median spread {spread.median():.2f} log units, max {spread.max():.2f}")
    print(f"  Taking the {policy}, not the best value — 'best ever' biases every "
          f"compound upward.")

agg['pIC50'] = 9.0 - np.log10(agg['ic50_nm'])

OUT_COLS = ['canonical_smiles', 'pIC50', 'ic50_nm', 'molecule_chembl_id',
            'assay_chembl_id', 'n_measurements']

full = agg[agg['ic50_nm'] <= CFG['training_max_ic50_nm']].copy()
full = full.sort_values('pIC50', ascending=False)
full_file = os.path.join(RESULTS_DIR, f'{TARGET_NAME}_bioactivity_full.csv')
full[OUT_COLS].to_csv(full_file, index=False)

# POTENCY_CUTOFF_NM drives the filter. It used to be defined, documented in the
# README as the knob to turn, and then ignored in favour of a hard-coded 50.0.
hits = agg[agg['ic50_nm'] < POTENCY_CUTOFF_NM].copy()
hits = hits.sort_values('pIC50', ascending=False)
if len(hits) == 0:
    raise RuntimeError(
        f"No compounds below {POTENCY_CUTOFF_NM} nM. Raise 'potency_cutoff_nm' "
        f"for target '{TARGET_NAME}' in pipeline_common.py (1000.0 is reasonable "
        f"for a hard target).")
hits_file = os.path.join(RESULTS_DIR, f'{TARGET_NAME}_inhibitors_cleaned.csv')
hits[OUT_COLS].to_csv(hits_file, index=False)

print(f"\n  full range   : {len(full):>5} compounds  "
      f"pIC50 {full['pIC50'].min():.2f}–{full['pIC50'].max():.2f}  -> ML training")
print(f"  potent hits  : {len(hits):>5} compounds  "
      f"pIC50 {hits['pIC50'].min():.2f}–{hits['pIC50'].max():.2f}  "
      f"(IC50 < {POTENCY_CUTOFF_NM} nM) -> SAR and seeds")

if full['pIC50'].std() < 0.8:
    print(f"  !! The full-range pIC50 spread is only {full['pIC50'].std():.2f} log "
          f"units. A regressor needs variance\n     to learn from; expect weak "
          f"predictions and treat Step 6 rankings as unreliable.")

# ==============================================================================
# 6. Distribution figure
# ==============================================================================
print("\n[6/6] Figure")
print("-" * 80)

fig, ax = plt.subplots(figsize=(10, 6))
ax.hist(full['pIC50'], bins=40, color='lightsteelblue', edgecolor='black',
        alpha=0.85, label=f'all curated (n={len(full)})')
ax.hist(hits['pIC50'], bins=40, color='steelblue', edgecolor='black',
        alpha=0.95, label=f'potent hits (n={len(hits)})')
ax.axvline(9.0 - np.log10(POTENCY_CUTOFF_NM), color='red', linestyle='--',
           linewidth=2, label=f'cutoff: IC50 = {POTENCY_CUTOFF_NM:g} nM')
ax.set_xlabel('pIC50 (−log10 M)', fontsize=12)
ax.set_ylabel('Frequency', fontsize=12)
title = f'{TARGET_NAME.upper()}'
if MUTANT_FILTER:
    title += f' ({MUTANT_FILTER})'
ax.set_title(f'pIC50 distribution — {title} inhibitors', fontsize=14, fontweight='bold')
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.legend()

hist_file = os.path.join(FIGURES_DIR, f'{TARGET_NAME}_pic50_distribution.png')
plt.savefig(hist_file, dpi=300, bbox_inches='tight')
plt.close()

print(f"  saved {hist_file}")
print("\n" + "=" * 80)
print("STEP 1 COMPLETE")
print("=" * 80)
print(f"  {full_file}")
print(f"  {hits_file}")
print(f"  {hist_file}")


In [ ]:
#!/usr/bin/env python3
"""
Step 2: Structure-Activity Relationship (SAR) Analysis
Analyzes the chemical space of inhibitors for the selected target
and identifies privileged scaffolds.
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
import sys
import os

warnings.filterwarnings('ignore')

# Set random seed for reproducibility
np.random.seed(42)

print("="*80)
print("Step 2: Structure-Activity Relationship (SAR) Analysis")
print("="*80)

# ==============================================================================
# CONFIGURATION — do not edit here; see pipeline_common.py
# ==============================================================================
# Configuration comes from pipeline_common.py via Step 0b, so this cell is
# byte-identical for KRAS, EGFR and mTOR. It is also safe to run standalone after a
# kernel restart: get_config() restores the selection from pipeline_config.json.
import pipeline_common
CFG = pipeline_common.get_config(quiet=True)
pipeline_common.export_globals(CFG, globals())
print(f"Target: {TARGET_NAME.upper()} ({MUTANT_FILTER or 'wild type'})")
# ==============================================================================

# Import RDKit
try:
    from rdkit import Chem
    from rdkit.Chem import Descriptors, Lipinski, QED, AllChem
    from rdkit.Chem.Scaffolds import MurckoScaffold
    from sklearn.decomposition import PCA
    print("✓ RDKit and scikit-learn imported successfully")
except ImportError as e:
    print(f"Error importing required libraries: {e}")
    print("Attempting to install missing packages...")
    import subprocess
    subprocess.run(["pip", "install", "-q", "rdkit", "scikit-learn"], check=True)
    from rdkit import Chem
    from rdkit.Chem import Descriptors, Lipinski, QED, AllChem
    from rdkit.Chem.Scaffolds import MurckoScaffold
    from sklearn.decomposition import PCA
    print("✓ Packages installed and imported successfully")

# BASE_DIR, RESULTS_DIR and FIGURES_DIR are already set by export_globals().
# Scaffold analysis uses the potent-hit set: "what do good compounds look like".
INPUT_FILENAME = f"{TARGET_NAME}_inhibitors_cleaned.csv"
INPUT_FILE = RESULTS_DIR / INPUT_FILENAME
OUTPUT_DIR = RESULTS_DIR

# ==============================================================================
# Step 1: Load Data
# ==============================================================================
print("\n" + "="*80)
print(f"Step 1: Loading cleaned {TARGET_NAME} dataset")
print("="*80)

if not INPUT_FILE.exists():
    print(f"❌ ERROR: Could not find input file: {INPUT_FILE}")
    print(f"   Please ensure Step 1 completed successfully and TARGET_NAME is correct.")
    sys.exit(1)

df = pd.read_csv(INPUT_FILE)
print(f"✓ Loaded {len(df)} compounds from {INPUT_FILE.name}")
print(f"  Columns: {list(df.columns)}")
print(f"  pIC50 range: {df['pIC50'].min():.2f} - {df['pIC50'].max():.2f}")

# ==============================================================================
# Step 2: Calculate Molecular Descriptors
# ==============================================================================
print("\n" + "="*80)
print("Step 2: Calculating Molecular Descriptors")
print("="*80)

def calculate_descriptors(smiles):
    """Calculate key physicochemical properties for a SMILES string."""
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            return None

        descriptors = {
            'MW': Descriptors.MolWt(mol),
            'LogP': Descriptors.MolLogP(mol),
            'TPSA': Descriptors.TPSA(mol),
            'HBD': Descriptors.NumHDonors(mol),
            'HBA': Descriptors.NumHAcceptors(mol),
            'QED': QED.qed(mol)
        }
        return descriptors
    except Exception as e:
        return None

# Calculate descriptors for all molecules
print("Calculating descriptors...")
descriptor_data = []
failed_count = 0

for i, smiles in enumerate(df['canonical_smiles']):
    if (i + 1) % 500 == 0:
        print(f"  Progress: {i + 1}/{len(df)} compounds processed ({100*(i+1)/len(df):.1f}%)")

    desc = calculate_descriptors(smiles)
    if desc is not None:
        descriptor_data.append(desc)
    else:
        failed_count += 1
        descriptor_data.append({
            'MW': np.nan, 'LogP': np.nan, 'TPSA': np.nan,
            'HBD': np.nan, 'HBA': np.nan, 'QED': np.nan
        })

# Add descriptors to dataframe
desc_df = pd.DataFrame(descriptor_data)
for col in desc_df.columns:
    df[col] = desc_df[col]

print(f"✓ Descriptors calculated for {len(df) - failed_count}/{len(df)} compounds")
if failed_count > 0:
    print(f"  Warning: {failed_count} compounds failed descriptor calculation")

# Print descriptor statistics
print("\nDescriptor Statistics:")
print(df[['MW', 'LogP', 'TPSA', 'HBD', 'HBA', 'QED']].describe())

# ==============================================================================
# Step 3: Scaffold Analysis (Bemis-Murcko)
# ==============================================================================
print("\n" + "="*80)
print("Step 3: Scaffold Analysis (Bemis-Murcko)")
print("="*80)

def get_bemis_murcko_scaffold(smiles):
    """Generate Bemis-Murcko scaffold SMILES."""
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            return None
        scaffold = MurckoScaffold.GetScaffoldForMol(mol)
        return Chem.MolToSmiles(scaffold)
    except:
        return None

# Generate scaffolds
print("Generating Bemis-Murcko scaffolds...")
scaffolds = []
scaffold_failed = 0

for i, smiles in enumerate(df['canonical_smiles']):
    if (i + 1) % 500 == 0:
        print(f"  Progress: {i + 1}/{len(df)} scaffolds generated ({100*(i+1)/len(df):.1f}%)")

    scaffold = get_bemis_murcko_scaffold(smiles)
    if scaffold is not None:
        scaffolds.append(scaffold)
    else:
        scaffolds.append(np.nan)
        scaffold_failed += 1

df['scaffold'] = scaffolds
print(f"✓ Scaffolds generated for {len(df) - scaffold_failed}/{len(df)} compounds")

# Identify most frequent scaffolds
scaffold_counts = df['scaffold'].value_counts()
print(f"\n✓ Identified {len(scaffold_counts)} unique scaffolds")
print(f"  Top 5 most frequent scaffolds:")
for i, (scaffold, count) in enumerate(scaffold_counts.head(5).items(), 1):
    print(f"    {i}. {scaffold[:50]}... (n={count})")

# Calculate average pIC50 for each scaffold
scaffold_stats = df.groupby('scaffold').agg({
    'pIC50': ['mean', 'std', 'count']
}).round(3)
scaffold_stats.columns = ['mean_pIC50', 'std_pIC50', 'count']
scaffold_stats = scaffold_stats.sort_values('mean_pIC50', ascending=False)

print(f"\n✓ Top 10 scaffolds by mean potency (pIC50):")
print(scaffold_stats.head(10))

# Save scaffold analysis
scaffold_summary = scaffold_stats.reset_index()
scaffold_output_file = OUTPUT_DIR / f"{TARGET_NAME}_scaffold_analysis.csv"
scaffold_summary.to_csv(scaffold_output_file, index=False)
MIN_COUNT = int(CFG['min_scaffold_count'])
scaffold_filtered = scaffold_stats.reset_index()
scaffold_filtered = scaffold_filtered[scaffold_filtered['count'] >= MIN_COUNT]
print(f"\n✓ Saved scaffold analysis to {scaffold_output_file.name}")
print(f"  ({len(scaffold_filtered)} scaffolds with >={MIN_COUNT} compounds, "
      f"out of {len(scaffold_stats)} unique scaffolds)")
if len(scaffold_filtered) == 0:
    print(f"  !! No scaffold reaches {MIN_COUNT} compounds, so there is no "
          f"recurring-series evidence here.")
    print(f"     Every 'privileged scaffold' below is a single compound. Lower "
          f"'min_scaffold_count' only if\n     you are willing to report it that way.")

# ==============================================================================
# Step 4: Chemical Space Visualization (FIXED)
# ==============================================================================
print("\n" + "="*80)
print("Step 4: Chemical Space Visualization (PCA on Morgan Fingerprints)")
print("="*80)

import numpy as np
from rdkit import Chem
from rdkit.Chem import DataStructs
from rdkit.Chem import rdFingerprintGenerator

def generate_morgan_fingerprint_numpy(smiles, radius=2, nBits=2048):
    """
    Generates a Morgan fingerprint and strictly converts it to a numpy array.
    Returns None if generation fails.
    """
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            return None

        # 1. Initialize the Generator (New Method)
        # This replaces AllChem.GetMorganFingerprintAsBitVect
        morgan_gen = rdFingerprintGenerator.GetMorganGenerator(radius=radius, fpSize=nBits)

        # 2. Generate Bit Vector
        fp_bitvect = morgan_gen.GetFingerprint(mol)

        # 3. Convert to Numpy Array (Standard RDKit method)
        fp_array = np.zeros((0,), dtype=np.int8)
        DataStructs.ConvertToNumpyArray(fp_bitvect, fp_array)

        return fp_array
    except Exception as e:
        # Print error for the first failure to help debug
        print(f"DEBUG: Fingerprint failed for {smiles[:10]}... Error: {e}")
        return None

# Generate fingerprints
print("Generating Morgan fingerprints (ECFP4, radius=2, 2048 bits)...")

valid_fingerprints = []
valid_indices = []
fp_failed = 0

for i, smiles in enumerate(df['canonical_smiles']):
    fp = generate_morgan_fingerprint_numpy(smiles)

    if fp is not None:
        valid_fingerprints.append(fp)
        valid_indices.append(i)
    else:
        fp_failed += 1

# Convert to Matrix
if len(valid_fingerprints) == 0:
    print("❌ CRITICAL ERROR: No valid fingerprints generated. Check your SMILES data.")
else:
    # Stack into a proper numpy matrix (Rows = compounds, Cols = bits)
    fingerprint_matrix = np.vstack(valid_fingerprints)
    print(f"✓ Fingerprints generated: shape {fingerprint_matrix.shape}")

    if fp_failed > 0:
        print(f"  Warning: {fp_failed} compounds failed and were dropped.")

    # SYNC DATAFRAME: Only keep rows where fingerprints succeeded
    df_clean = df.iloc[valid_indices].copy().reset_index(drop=True)
    print(f"✓ Created cleaned dataframe with {len(df_clean)} compounds")

    # Perform PCA
    print("\nPerforming PCA (2 components)...")
    from sklearn.decomposition import PCA

    # Verify we have variance (cannot run PCA on identical rows)
    if np.var(fingerprint_matrix) == 0:
         print("❌ Error: Zero variance in fingerprints (all molecules identical?). Cannot run PCA.")
    else:
        pca = PCA(n_components=2, random_state=42)
        pca_coords = pca.fit_transform(fingerprint_matrix)

        print(f"✓ PCA completed")
        print(f"  Explained variance: PC1={pca.explained_variance_ratio_[0]:.3f}, PC2={pca.explained_variance_ratio_[1]:.3f}")
        print(f"  Total variance explained: {sum(pca.explained_variance_ratio_):.3f}")

        # Add PCA coordinates to the CLEAN dataframe
        df_clean['PC1'] = pca_coords[:, 0]
        df_clean['PC2'] = pca_coords[:, 1]

        # Update the main df to be the clean version for subsequent steps
        df = df_clean
        print(f"✓ Updated main dataframe with PCA coordinates")

# ==============================================================================
# Step 5: Generate Visualizations
# ==============================================================================
print("\n" + "="*80)
print("Step 5: Generating Visualizations")
print("="*80)

# Set matplotlib style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# Construct Title Prefix
plot_title_prefix = f"{TARGET_NAME.upper()}"
if MUTANT_FILTER:
    plot_title_prefix += f" ({MUTANT_FILTER})"

# --- Figure 1: Chemical Space PCA ---
print("Creating Figure 1: Chemical Space PCA...")
fig, ax = plt.subplots(figsize=(10, 8))

scatter = ax.scatter(df['PC1'], df['PC2'], c=df['pIC50'],
                     cmap='viridis', s=30, alpha=0.6, edgecolors='none')

cbar = plt.colorbar(scatter, ax=ax)
cbar.set_label('pIC50', fontsize=12, weight='bold')

ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)',
              fontsize=12, weight='bold')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)',
              fontsize=12, weight='bold')
ax.set_title(f'Chemical Space of {plot_title_prefix} Inhibitors\n(PCA on Morgan Fingerprints)',
             fontsize=14, weight='bold', pad=20)

ax.grid(True, alpha=0.3)
plt.tight_layout()

pca_fig_file = FIGURES_DIR / f"{TARGET_NAME}_chemical_space_pca.png"
plt.savefig(pca_fig_file, dpi=300, bbox_inches='tight')
plt.close()
print(f"✓ Saved: {pca_fig_file.name}")

# --- Figure 2: Physicochemical Properties Distribution ---
print("Creating Figure 2: Physicochemical Properties...")
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

properties = ['MW', 'LogP', 'TPSA', 'HBD', 'HBA', 'QED']
colors = plt.cm.Set2(range(len(properties)))

for i, prop in enumerate(properties):
    ax = axes[i]

    # Create violin plot
    parts = ax.violinplot([df[prop].dropna()], positions=[0],
                          showmeans=True, showmedians=True, widths=0.7)

    # Color the violin
    for pc in parts['bodies']:
        pc.set_facecolor(colors[i])
        pc.set_alpha(0.7)

    # Add box plot overlay
    bp = ax.boxplot([df[prop].dropna()], positions=[0], widths=0.3,
                    patch_artist=True, showfliers=False)
    bp['boxes'][0].set_facecolor(colors[i])
    bp['boxes'][0].set_alpha(0.5)

    # Labels
    ax.set_title(prop, fontsize=12, weight='bold')
    ax.set_ylabel('Value', fontsize=10)
    ax.set_xticks([])
    ax.grid(True, alpha=0.3, axis='y')

    # Add statistics
    mean_val = df[prop].mean()
    median_val = df[prop].median()
    ax.text(0.02, 0.98, f'Mean: {mean_val:.2f}\nMedian: {median_val:.2f}',
            transform=ax.transAxes, fontsize=9,
            verticalalignment='top', bbox=dict(boxstyle='round',
            facecolor='white', alpha=0.8))

fig.suptitle(f'Distribution of Physicochemical Properties\n({plot_title_prefix} Inhibitor Dataset)',
             fontsize=14, weight='bold', y=0.995)
plt.tight_layout()

props_fig_file = FIGURES_DIR / f"{TARGET_NAME}_physicochemical_properties.png"
plt.savefig(props_fig_file, dpi=300, bbox_inches='tight')
plt.close()
print(f"✓ Saved: {props_fig_file.name}")

# --- Figure 3: Top Scaffolds by Potency ---
print("Creating Figure 3: Top Scaffolds by Potency...")

top_scaffolds = scaffold_filtered.head(20)

fig, ax = plt.subplots(figsize=(12, 8))

if not top_scaffolds.empty:
    # Create bar plot
    bars = ax.barh(range(len(top_scaffolds)), top_scaffolds['mean_pIC50'],
                   color=plt.cm.viridis(np.linspace(0.3, 0.9, len(top_scaffolds))))

    # Add error bars
    ax.errorbar(top_scaffolds['mean_pIC50'], range(len(top_scaffolds)),
                xerr=top_scaffolds['std_pIC50'], fmt='none', ecolor='black',
                capsize=3, alpha=0.5, linewidth=1)

    # Add count labels
    for i, (idx, row) in enumerate(top_scaffolds.iterrows()):
        ax.text(row['mean_pIC50'] + 0.1, i, f"n={int(row['count'])}",
                va='center', fontsize=9, weight='bold')

    # Create scaffold labels (truncated)
    labels = []
    for scaffold in top_scaffolds['scaffold']:
        if len(scaffold) > 40:
            label = scaffold[:37] + "..."
        else:
            label = scaffold
        labels.append(label)

    ax.set_yticks(range(len(top_scaffolds)))
    ax.set_yticklabels(labels, fontsize=8, family='monospace')
    ax.set_xlabel('Mean pIC50', fontsize=12, weight='bold')
    ax.set_title(f'Most potent Scaffolds for {plot_title_prefix}\n(Scaffolds with ≥5 compounds)',
                 fontsize=14, weight='bold', pad=20)
    ax.grid(True, alpha=0.3, axis='x')
    ax.invert_yaxis()
else:
    ax.text(0.5, 0.5, "Not enough data for scaffold analysis", ha='center', fontsize=14)

plt.tight_layout()

scaffolds_fig_file = FIGURES_DIR / f"{TARGET_NAME}_top_scaffolds_potency.png"
plt.savefig(scaffolds_fig_file, dpi=300, bbox_inches='tight')
plt.close()
print(f"✓ Saved: {scaffolds_fig_file.name}")

# ==============================================================================
# Step 6: Save Augmented Dataset
# ==============================================================================
print("\n" + "="*80)
print("Step 6: Saving Augmented Dataset")
print("="*80)

sar_output_file = OUTPUT_DIR / f"{TARGET_NAME}_sar_analysis.csv"
df.to_csv(sar_output_file, index=False)
print(f"✓ Saved augmented dataset to {sar_output_file.name}")
print(f"  Columns: {list(df.columns)}")
print(f"  Shape: {df.shape}")

# ==============================================================================
# Summary Statistics
# ==============================================================================
print("\n" + "="*80)
print("SUMMARY")
print("="*80)

print("\nDataset Characteristics:")
print(f"  Target: {TARGET_NAME.upper()}")
print(f"  Total compounds: {len(df)}")
print(f"  Unique scaffolds: {len(scaffold_counts)}")
print(f"  Scaffolds with ≥5 compounds: {len(scaffold_filtered)}")

print("\nMolecular Property Ranges:")
print(f"  MW: {df['MW'].min():.1f} - {df['MW'].max():.1f} Da")
print(f"  LogP: {df['LogP'].min():.2f} - {df['LogP'].max():.2f}")
print(f"  TPSA: {df['TPSA'].min():.1f} - {df['TPSA'].max():.1f} Ų")
print(f"  HBD: {df['HBD'].min():.0f} - {df['HBD'].max():.0f}")
print(f"  HBA: {df['HBA'].min():.0f} - {df['HBA'].max():.0f}")
print(f"  QED: {df['QED'].min():.3f} - {df['QED'].max():.3f}")

print("\nLipinski's Rule of Five Compliance:")
ro5_pass = ((df['MW'] <= 500) &
            (df['LogP'] <= 5) &
            (df['HBD'] <= 5) &
            (df['HBA'] <= 10)).sum()
print(f"  Compounds passing Ro5: {ro5_pass}/{len(df)} ({100*ro5_pass/len(df):.1f}%)")

print("\nOutput Files:")
print(f"  ✓ {sar_output_file.name}")
print(f"  ✓ {scaffold_output_file.name}")
print(f"  ✓ {pca_fig_file.name}")
print(f"  ✓ {props_fig_file.name}")
print(f"  ✓ {scaffolds_fig_file.name}")

print("\n" + "="*80)
print("SAR Analysis Complete!")
print("="*80)

In [ ]:
#!/usr/bin/env python3
"""
Step 3: Genomic & Structural Context Analysis (Final Polished Version)
Target: Any specified target (e.g., EGFR T790M, KRAS G12D)
Features: Robust Search + Detailed Reporting + Text File Output
"""

import sys
import json
import requests
import time
from pathlib import Path
from typing import Dict, List, Tuple, Optional
import numpy as np
import pandas as pd
import warnings
import re
import subprocess

# Bio module imports
try:
    from Bio import Entrez, PDB
    from Bio.PDB import PDBParser, PDBIO, Select
except ImportError:
    print("Installing Biopython...")
    import subprocess
    subprocess.run(["pip", "install", "-q", "biopython"], check=True)
    from Bio import Entrez, PDB
    from Bio.PDB import PDBParser, PDBIO, Select

# ChEMBL Import (For UniProt lookup)
try:
    from chembl_webresource_client.new_client import new_client
except ImportError:
    print("Installing ChEMBL Client...")
    subprocess.run(["pip", "install", "-q", "chembl_webresource_client"], check=True)
    from chembl_webresource_client.new_client import new_client

warnings.filterwarnings('ignore')

# 🛠️ MANUAL PDB OVERRIDE
# Set this to a string (e.g., "4JT6") to force a specific structure.
# Set to None to enable automatic searching.
# ------------------------------------------------------------------------------
# ==============================================================================
# Configuration from pipeline_common.py -- identical cell for every target.
# MANUAL_PDB_ID, the search term and the mutant filter all come from the config.
# ==============================================================================
import pipeline_common
CFG = pipeline_common.get_config(quiet=True)
pipeline_common.export_globals(CFG, globals())

# NCBI requires a real contact address and throttles or blocks invalid ones. The
# address used to be hard-coded here, overriding whatever the user had set.
Entrez.email = CFG['entrez_email']
if str(Entrez.email).startswith('your.email'):
    print("!! 'entrez_email' is still the placeholder. Set it to your own address in")
    print("   pipeline_common.py before relying on the PubMed search -- NCBI blocks")
    print("   anonymous bulk queries.")

print("=" * 80)
print(f"Step 3: {PROTEIN_SEARCH_TERM} {MUTANT_FILTER if MUTANT_FILTER else 'WT'} Target Analysis")
print("=" * 80)


# ==============================================================================
# 🧬 AUTOMATED ID RETRIEVAL LOGIC (The Fix)
# ==============================================================================

def fetch_uniprot_id_from_api(gene_name: str) -> Optional[str]:
    """
    Query the UniProt API directly if ChEMBL fails.
    Search for: Gene Name + Human + Reviewed (Swiss-Prot)
    """
    print(f"  ℹ️ Querying UniProt API for gene: '{gene_name}' (Homo sapiens)...")

    # UniProt REST API Query
    # query = gene:{NAME} AND organism_id:9606 (Human) AND reviewed:true (Swiss-Prot only)
    url = "https://rest.uniprot.org/uniprotkb/search"
    params = {
        "query": f"(gene_exact:{gene_name}) AND (organism_id:9606) AND (reviewed:true)",
        "format": "json",
        "size": 1,  # We only want the top hit (the main protein)
        "fields": "accession,id,protein_name"
    }

    try:
        response = requests.get(url, params=params)
        if response.status_code == 200:
            results = response.json().get('results', [])
            if results:
                top_hit = results[0]
                acc_id = top_hit['primaryAccession']
                name = top_hit['uniProtkbId']
                print(f"  ✓ UniProt API found: {name} ({acc_id})")
                return acc_id
            else:
                print("  ⚠️ UniProt API returned no results.")
        else:
            print(f"  ⚠️ UniProt API Error: {response.status_code}")
    except Exception as e:
        print(f"  ⚠️ Network Error connecting to UniProt: {e}")

    return None

def get_target_uniprot_auto(target_chembl_id: str, gene_name: str) -> Optional[str]:
    """Smart Manager: Tries ChEMBL first, then fails over to UniProt API."""

    # Method 1: Try ChEMBL (Fastest, but sometimes empty)
    if target_chembl_id:
        print(f"\n[1/7] Identifying Target Genetic Fingerprint (UniProt ID)...")
        print(f"  Attempt 1: Checking ChEMBL ({target_chembl_id})...")
        try:
            target_data = new_client.target.get(target_chembl_id)
            for xref in target_data.get('cross_references', []):
                if xref['xref_src'] == 'uniprot':
                    uniprot_id = xref['xref_id']
                    print(f"  ✓ Found in ChEMBL: {uniprot_id}")
                    return uniprot_id
        except:
            print("  ⚠️ ChEMBL lookup failed.")

    # Method 2: Try UniProt API (Robust Fallback)
    print(f"  Attempt 2: Checking UniProt Database directly...")
    api_id = fetch_uniprot_id_from_api(gene_name)
    if api_id:
        return api_id

    print("❌ Could not determine UniProt ID automatically.")
    return None


def fetch_pdbs_by_uniprot(uniprot_id: str) -> List[Dict]:
    """Searches RCSB PDB for structures matching the EXACT UniProt ID."""
    print(f"  Searching PDB for structures containing UniProt sequence: {uniprot_id}...")

    # Query: (Contains UniProt ID) AND (Has Non-Polymer Ligand)
    query = {
        "query": {
            "type": "group",
            "logical_operator": "and",
            "nodes": [
                {
                    "type": "terminal",
                    "service": "text",
                    "parameters": {
                        "attribute": "rcsb_polymer_entity_container_identifiers.reference_sequence_identifiers.database_accession",
                        "operator": "exact_match",
                        "value": uniprot_id
                    }
                },
                {
                    "type": "terminal",
                    "service": "text",
                    "parameters": {
                        "attribute": "rcsb_entry_info.nonpolymer_entity_count",
                        "operator": "greater",
                        "value": 0
                    }
                }
            ]
        },
        "request_options": {"return_all_hits": True},
        "return_type": "entry"
    }

    try:
        response = requests.post("https://search.rcsb.org/rcsbsearch/v2/query", json=query)
        if response.status_code == 200:
            result_ids = response.json().get('result_set', [])
            print(f"  ✓ Found {len(result_ids)} structures matching {uniprot_id}.")
            return [{"identifier": pid} for pid in result_ids]
    except Exception as e:
        print(f"  ⚠️ UniProt Search Error: {e}")
    return []


def get_pdb_details(pdb_ids: List[str]) -> List[Dict]:
    """
    Get detailed information - PERMISSIVE MODE.
    1. Fixes the 'Dictionary vs String' bug causing 0 results.
    2. Removes ALL 'drug-like' filters to accept any structure.
    """
    print(f"\n[2/7] Retrieving PDB structure details (Scanning top {len(pdb_ids)} candidates)...")
    structures = []
    ids_to_check = pdb_ids

    # Only filter out pure water. Accept everything else (ions, small molecules).
    JUNK_LIGANDS = ["HOH", "WAT"]

    for i, raw_id in enumerate(ids_to_check):
        # 🛠️ CRITICAL FIX: Handle case where ID is passed as a dictionary
        if isinstance(raw_id, dict):
            pdb_id = raw_id.get('identifier', str(raw_id))
        else:
            pdb_id = str(raw_id)

        if i % 10 == 0: print(f"  Processing {i+1}/{len(ids_to_check)}: {pdb_id}")

        data_url = f"https://data.rcsb.org/rest/v1/core/entry/{pdb_id}"
        try:
            response = requests.get(data_url)
            if response.status_code == 200:
                data = response.json()

                # 1. Get Resolution
                resolution = 99.9
                if "rcsb_entry_info" in data:
                    res_list = data["rcsb_entry_info"].get("resolution_combined", [])
                    if res_list and res_list[0] is not None:
                        resolution = float(res_list[0])

                # 2. Get Ligands (NO RESTRICTIONS)
                ligands = []
                if "pdbx_entity_nonpoly" in data:
                    for entity in data["pdbx_entity_nonpoly"]:
                        comp_id = entity.get("comp_id")
                        name = entity.get("name", "").lower()

                        # Accept ANYTHING that isn't water
                        if comp_id and comp_id not in JUNK_LIGANDS and "water" not in name:
                            ligands.append(comp_id)

                # 3. Always add the structure, even if no ligands found
                structures.append({
                    "pdb_id": pdb_id,
                    "resolution": resolution,
                    "has_ligand": len(ligands) > 0,
                    "ligands": list(set(ligands))
                })
        except Exception as e:
            print(f"  ⚠️ Failed to fetch {pdb_id}: {e}")
            pass

    # Summary
    valid_count = len([s for s in structures if s['has_ligand']])
    print(f"  ✓ Scanning complete. Retrieved {len(structures)} structures ({valid_count} with ligands).")

    return structures


def select_best_structure(structures: List[Dict]) -> Optional[str]:
    """Select best structure with actionable warnings for Step 5."""
    print("\n[3/7] Selecting best structure...")

    # Priority 1: Valid Ligand + Good Resolution
    valid = [s for s in structures if s['has_ligand'] and s['resolution'] < 10.0]

    # Priority 2: Fallback (Best Resolution only)
    is_fallback = False
    if not valid:
        print("  ⚠️ No structure with a relevant DRUG-LIKE ligand found.")
        print("  ⚠️ Falling back to best resolution available.")
        valid = structures
        is_fallback = True

    if not valid: return None

    # Sort by Resolution
    valid.sort(key=lambda x: x['resolution'])
    best = valid[0]

    print(f"\n  🏆 Selected: {best['pdb_id']}")
    print(f"    Resolution: {best['resolution']:.2f} Å")

    if is_fallback:
        print(f"    Ligands: None (or ions only)")
        print("\n" + "-"*60)
        print("⚠️  WARNING FOR STEP 5  ⚠️")
        print("!"*60)
        print(f"  Since PDB {best['pdb_id']} might not have a reference ligand, the docking")
        print("  grid box might default to the PROTEIN CENTER (Belly Button).")
        print("  ")
        print("  YOU WILL NEED TO MANUALLY SET THE ACTIVE SITE RESIDUE IN STEP 5 IF YOU STILL DON'T SEE A LIGAND BELOW:")
        print(f"  1. Look up the active site residue number for {best['pdb_id']}.")
        print("  2. In Step 5, set: TARGET_RESIDUE_ID = <your_residue_number>")
        print("-"*60 + "\n")
    else:
        print(f"    Relevant Ligands: {', '.join(best['ligands'])}")

    return best['pdb_id']


def download_pdb_structure(pdb_id: str) -> Path:
    print(f"\n[4/7] Downloading PDB structure {pdb_id}...")
    output_path = WORKFLOW_DATA_DIR / f"{pdb_id}.pdb"
    if output_path.exists():
        print(f"  ✓ File already exists: {output_path}")
        return output_path

    response = requests.get(f"https://files.rcsb.org/download/{pdb_id}.pdb")
    if response.status_code == 200:
        with open(output_path, 'w') as f: f.write(response.text)
        print(f"  Downloaded to: {output_path}")
        return output_path
    else:
        raise Exception(f"Failed to download {pdb_id}")


def analyze_structure(pdb_path: Path, pdb_id: str) -> Dict:
    """Analyze PDB (Detailed Output)."""
    print(f"\n[5/7] Analyzing structure {pdb_id}...")

    parser = PDBParser(QUIET=True)
    structure = parser.get_structure(pdb_id, pdb_path)
    model = structure[0]

    # 1. Find Mutant
    target_res_found = None
    chain_id = None
    mutation_position = None
    search_pos = None

    if MUTANT_FILTER:
        try:
            match = re.search(r'\d+', MUTANT_FILTER)
            if match: search_pos = int(match.group())
        except: pass

    for chain in model:
        for residue in chain:
            if search_pos and residue.get_id()[1] == search_pos:
                target_res_found = residue
                chain_id = chain.get_id()
                mutation_position = residue.get_id()[1]
                print(f"  Found residue {residue.get_resname()} at position {mutation_position} in chain {chain_id}")
                break
        if target_res_found: break

    # 2. Find Ligands
    ligands = [r for r in model.get_residues() if r.get_id()[0].startswith("H_")
               and r.get_resname() not in ["HOH", "WAT", "SO4", "PO4", "EDO", "DMS", "MG", "NA", "CL"]]

    ligand_names = [l.get_resname() for l in ligands]
    print(f"  Found {len(ligands)} ligand(s): {list(set(ligand_names))}")

    pocket_residues = []
    min_dist = 999.0
    closest_lig_name = "None"

    if ligands:
        main_lig = ligands[0]
        closest_lig_name = main_lig.get_resname()
        atoms_lig = list(main_lig.get_atoms())
        atoms_prot = [a for c in model for r in c for a in r.get_atoms() if not r.get_id()[0].startswith("H_")]

        ns = PDB.NeighborSearch(atoms_prot)
        for atom in atoms_lig:
            neighbors = ns.search(atom.get_coord(), 5.0, level='R')
            pocket_residues.extend(neighbors)

        if target_res_found:
            for a1 in target_res_found.get_atoms():
                for a2 in atoms_lig:
                    dist = a1 - a2
                    if dist < min_dist: min_dist = dist

        # Format for output
        site_label = MUTANT_FILTER if MUTANT_FILTER else "Target Site"
        if min_dist < 999:
            print(f"  {site_label} to ligand ({closest_lig_name}) distance: {min_dist:.2f} Å")
        else:
             print(f"  {site_label} to ligand distance: N/A (too far or not found)")

    pocket_residues = list(set(pocket_residues))
    print(f"  Identified {len(pocket_residues)} binding pocket residues")

    final_dist = float(min_dist) if min_dist < 999 else None

    return {
        "pdb_id": pdb_id,
        "mutation_site": {
            "residue": target_res_found.get_resname() if target_res_found else "Unknown",
            "position": int(mutation_position) if mutation_position else "Unknown",
            "chain": chain_id if chain_id else "Unknown"
        },
        "distance_to_ligand": round(final_dist, 2) if final_dist else None,
        "closest_ligand": closest_lig_name,
        "pocket_residue_count": int(len(pocket_residues))
    }


def mine_pubmed_literature() -> List[Dict]:
    """Search PubMed with Detailed Output."""
    print("\n[6/7] Mining PubMed literature...")

    search_terms = [f"{PROTEIN_SEARCH_TERM} resistance mechanisms"]
    if MUTANT_FILTER:
        search_terms.append(f"{PROTEIN_SEARCH_TERM} {MUTANT_FILTER} structure drug")

    abstracts = []
    total_found = 0

    for term in search_terms:
        print(f"\n  Searching: '{term}'")
        try:
            handle = Entrez.esearch(db="pubmed", term=term, retmax=5, sort="relevance")
            record = Entrez.read(handle)
            handle.close()
            ids = record["IdList"]

            count = len(ids)
            print(f"  Found {count} articles")
            total_found += count

            if ids:
                handle = Entrez.efetch(db="pubmed", id=ids, rettype="abstract", retmode="xml")
                records = Entrez.read(handle)
                handle.close()
                for art in records['PubmedArticle']:
                    try:
                        title = art['MedlineCitation']['Article']['ArticleTitle']
                        pmid = str(art['MedlineCitation']['PMID'])
                        # Get abstract text
                        abst_text = ""
                        if 'Abstract' in art['MedlineCitation']['Article']:
                             abst_list = art['MedlineCitation']['Article']['Abstract']['AbstractText']
                             abst_text = " ".join([str(x) for x in abst_list])

                        abstracts.append({
                            "pmid": pmid,
                            "title": title,
                            "abstract": abst_text,
                            "query": term
                        })
                    except: pass
        except: pass

    print(f"\n  Total abstracts retrieved: {len(abstracts)}")
    return abstracts


def save_results_to_files(analysis, abstracts):
    """Save results with restored functionality."""
    print("\n[7/7] Saving results...")

    # 1. JSON
    out_json = RESULTS_DIR / f"{TARGET_NAME}_structural_analysis.json"
    with open(out_json, 'w') as f:
        json.dump({"structure": analysis, "literature": abstracts}, f, indent=2)
    print(f"  Saved: {out_json}")

    # 2. Text File (Restored feature)
    out_txt = RESULTS_DIR / f"{TARGET_NAME}_literature_findings.txt"
    with open(out_txt, 'w') as f:
        f.write("=" * 80 + "\n")
        f.write(f"{PROTEIN_SEARCH_TERM} {MUTANT_FILTER if MUTANT_FILTER else 'WT'} Analysis Findings\n")
        f.write("=" * 80 + "\n\n")
        for i, ab in enumerate(abstracts, 1):
            f.write(f"[{i}] PMID: {ab['pmid']}\n")
            f.write(f"Title: {ab['title']}\n")
            f.write(f"Query: {ab['query']}\n")
            f.write(f"Abstract: {ab['abstract'][:300]}...\n\n") # Truncate for readability
    print(f"  Saved: {out_txt}")


def main():
    try:
        best_id = None
        uniprot_id = "Manual Input"

        # ----------------------------------------------------------------------
        # IF STATEMENT FOR MANUAL PDB INPUT (Requested Logic)
        # ----------------------------------------------------------------------
        if MANUAL_PDB_ID is not None:
            print(f"\nℹ️ MANUAL MODE ACTIVE: Skipping automated search.")
            print(f"      Using Manual PDB ID: {MANUAL_PDB_ID}")
            best_id = MANUAL_PDB_ID

        else:
            # DEFAULT: Perform Automated Search if PDB = None
            uniprot_id = get_target_uniprot_auto(TARGET_CHEMBL_ID, PROTEIN_SEARCH_TERM)
            if not uniprot_id:
                print("❌ CRITICAL: Could not find UniProt ID. Cannot proceed safely.")
                return

            structures = fetch_pdbs_by_uniprot(uniprot_id)
            if not structures:
                print(f"❌ No PDB structures found for UniProt ID {uniprot_id}")
                return

            pdb_ids = [s['identifier'] for s in structures]
            details = get_pdb_details(pdb_ids)
            best_id = select_best_structure(details)

        # ----------------------------------------------------------------------

        if not best_id: return

        pdb_path = download_pdb_structure(best_id)
        analysis = analyze_structure(pdb_path, best_id)
        lit = mine_pubmed_literature()
        save_results_to_files(analysis, lit)

        print("\n" + "="*80 + "\nANALYSIS COMPLETE\n" + "="*80)
        print(f"Target: {PROTEIN_SEARCH_TERM} ({uniprot_id})")
        print(f"Selected PDB: {analysis['pdb_id']}")
        print(f"Ligand: {analysis['closest_ligand']}")

    except Exception as e:
        print(f"\n❌ ERROR: {e}")
        import traceback
        traceback.print_exc()

if __name__ == "__main__":
    main()

In [ ]:
#!/usr/bin/env python3
"""
Step 4: Analogs Generation of Novel Inhibitor (Generalized)

This script generates novel chemical entities based on high-potency inhibitors
identified in the SAR analysis (Step 2). It uses chemical mutation strategies
to explore the chemical space around potent scaffolds, applies drug-likeness filters,
and selects top candidates for downstream virtual screening.

Features:
- Target-Agnostic: Works for EGFR, KRAS, BRAF, etc.
- Smart Configuration: Auto-detects input files from previous steps.
- RDKit Validation: Filters out disconnected fragments and dummy atoms.
- SA Score: Ranking by Synthetic Accessibility.
"""

import pandas as pd
import numpy as np
from rdkit import Chem
from rdkit.Chem import AllChem, Descriptors, Crippen, QED
from rdkit.Chem import rdMolDescriptors, Draw
import random
from collections import defaultdict
from typing import List, Tuple, Set
import time
import sys
import os
import subprocess
import gzip
import pickle
import math
from pathlib import Path
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from rdkit.Chem import rdFingerprintGenerator

# Suppress RDKit warnings for cleaner output
from rdkit import RDLogger
RDLogger.DisableLog('rdApp.*')

# Set random seeds for reproducibility
random.seed(42)
np.random.seed(42)

# ==============================================================================
# CONFIGURATION — do not edit here; see pipeline_common.py
# ==============================================================================
import pipeline_common
CFG = pipeline_common.get_config(quiet=True)
pipeline_common.export_globals(CFG, globals())

print("="*80)
print(f"STEP 4: ANALOGS GENERATION FOR {TARGET_NAME.upper()}")
print("="*80)

# BASE_DIR, RESULTS_DIR and FIGURES_DIR come from the config.

# Input Files (Dynamic based on TARGET_NAME)
SCAFFOLD_FILE = RESULTS_DIR / f"{TARGET_NAME}_scaffold_analysis.csv"
SAR_FILE = RESULTS_DIR / f"{TARGET_NAME}_sar_analysis.csv"
SEEDS_FILE = RESULTS_DIR / f"{TARGET_NAME}_selected_seeds.csv"

# Output Files
CANDIDATES_FILE = RESULTS_DIR / f"{TARGET_NAME}_generated_candidates.csv"
TOP20_CANDIDATES_FILE = RESULTS_DIR / f"{TARGET_NAME}_top20_generated_candidates.csv"
PCA_FIG_FILE = FIGURES_DIR / f"{TARGET_NAME}_generation_pca.png"
TSNE_FIG_FILE = FIGURES_DIR / f"{TARGET_NAME}_generation_tsne.png"

# Check if inputs exist
if not SCAFFOLD_FILE.exists() or not SAR_FILE.exists():
    print(f"❌ ERROR: Input files not found for {TARGET_NAME}.")
    print(f"   Missing: {SCAFFOLD_FILE}")
    print(f"   Missing: {SAR_FILE}")
    print("   Please ensure Step 2 (SAR Analysis) completed successfully.")
    sys.exit(1)

# ==============================================================================
# 📥 DOWNLOAD SA SCORE DEPENDENCIES & INITIALIZE (FINAL FIX)
# ==============================================================================
print("\n[0/6] Setting up Synthetic Accessibility (SA) Scorer...")
print("🔄 RESETTING SA SCORING ENVIRONMENT...")

# 1. DELETE OLD FILE & RE-DOWNLOAD
filename = "fpscores.pkl.gz"
if os.path.exists(filename):
    os.remove(filename)
    print(f"  🗑️ Deleted old {filename}")

url = "https://raw.githubusercontent.com/rdkit/rdkit/master/Contrib/SA_Score/fpscores.pkl.gz"
print(f"  ⬇️ Downloading fresh {filename}...")
subprocess.run(["wget", "-O", filename, url], capture_output=True)

# 2. LOAD DATA (FIXED FOR RDKit FORMAT)
SA_dictionary = {}

if os.path.exists(filename):
    try:
        with gzip.open(filename, 'rb') as f:
            raw_obj = pickle.load(f, encoding='latin1')

        print(f"  🔍 Fresh file loaded. Raw Type: {type(raw_obj)}")

        # --- LOGIC UPDATE: Handle [[score, fp1, fp2...], ...] ---
        if isinstance(raw_obj, list):
            print(f"  ⚠ Loaded as LIST (len={len(raw_obj)}). Unpacking RDKit format...")

            try:
                # The RDKit format is: [[score, fp_id, fp_id...], [score, fp_id...]]
                # Index 0 is the score. Indices 1->End are the fingerprints.
                for entry in raw_obj:
                    score = float(entry[0]) # First item is the score
                    for fp_id in entry[1:]: # Rest are the IDs
                        SA_dictionary[fp_id] = score

                print(f"  ✓ Success! Unpacked {len(SA_dictionary)} fragment scores.")

            except Exception as e:
                 print(f"  ❌ Failed to unpack list: {e}")

        elif isinstance(raw_obj, dict):
            SA_dictionary = raw_obj
            print(f"  ✓ Success! Object was already a dictionary.")

    except Exception as e:
        print(f"  ❌ File load failed: {e}")
else:
    print("  ❌ Download failed.")

# 3. DEFINE SCORING FUNCTION
def calculate_sa_clean(m, score_dict):
    if not score_dict: return 5.0

    fp = AllChem.GetMorganFingerprint(m, 2)
    fps = fp.GetNonzeroElements()
    score1 = 0.
    nf = 0

    for bitId, v in fps.items():
        nf += v
        score1 += score_dict.get(bitId, -4) * v

    if nf == 0: return 5.0
    score1 /= nf

    nAtoms = m.GetNumAtoms()
    nChiralCenters = len(Chem.FindMolChiralCenters(m, includeUnassigned=True))
    ri = m.GetRingInfo()
    nBridgeheads = rdMolDescriptors.CalcNumBridgeheadAtoms(m)
    nSpiro = rdMolDescriptors.CalcNumSpiroAtoms(m)
    nMacrocycles = 0
    for x in ri.AtomRings():
        if len(x) > 8: nMacrocycles += 1

    sizePenalty = nAtoms**1.005 - nAtoms
    stereoPenalty = math.log10(nChiralCenters + 1)
    spiroPenalty = math.log10(nSpiro + 1)
    bridgePenalty = math.log10(nBridgeheads + 1)
    macrocyclePenalty = 0.
    if nMacrocycles > 0: macrocyclePenalty = math.log10(2)

    score2 = 0. - sizePenalty - stereoPenalty - spiroPenalty - bridgePenalty - macrocyclePenalty

    score3 = 0.
    if nAtoms > len(fps):
        score3 = math.log(float(nAtoms) / len(fps)) * .5

    sascore = score1 + score2 + score3

    min_sa = -4.0
    max_sa = 2.5
    sascore = 11. - (sascore - min_sa + 1) / (max_sa - min_sa) * 9.

    if sascore > 10.: sascore = 10.0
    elif sascore < 1.: sascore = 1.0

    return sascore

# ============================================================================
# 1. LOAD SEED MOLECULES
# ============================================================================

print("\n[1/6] Loading seed molecules from SAR analysis...")

if not SCAFFOLD_FILE.exists():
    print(f"❌ Input file missing: {SCAFFOLD_FILE}")
    sys.exit(1)

scaffold_df = pd.read_csv(SCAFFOLD_FILE)

# --- HYBRID STRATEGY (thresholds from the shared config) ---
MIN_COUNT = int(CFG['min_scaffold_count'])

# 1. Proven scaffolds (exploitation): recurring series, ranked by how often they recur.
proven_df = (scaffold_df[scaffold_df['count'] >= MIN_COUNT]
             .sort_values('count', ascending=False)
             .head(int(CFG['n_seeds_proven'])))
proven_df['strategy'] = 'Proven (Exploitation)'

# 2. Exploration scaffolds (moonshot): seen fewer than MIN_COUNT times, ranked by
#    mean potency. A scaffold with count == 1 has a mean of one measurement and an
#    undefined standard deviation -- treat its "potency" as a single data point.
exploration_df = (scaffold_df[scaffold_df['count'] < MIN_COUNT]
                  .sort_values('mean_pIC50', ascending=False)
                  .head(int(CFG['n_seeds_exploration'])))
exploration_df['strategy'] = 'Exploration (Moonshot)'
n_singletons = int((exploration_df['count'] == 1).sum())
if n_singletons:
    print(f"  Note: {n_singletons}/{len(exploration_df)} exploration seeds rest on a "
          f"single measurement.")

# Combine
combined_df = pd.concat([proven_df, exploration_df]).drop_duplicates(subset=['scaffold'])

# Save Selected Seeds
combined_df.to_csv(SEEDS_FILE, index=False)

print(f"  ✓ Strategy Applied: Hybrid Portfolio (Top 20)")
print(f"    - Proven (Robust SAR): {len(proven_df)}")
print(f"    - Exploration (High Potency): {len(exploration_df)}")
print(f"  ✓ Saved selected seeds to {SEEDS_FILE}")

seed_smiles = combined_df['scaffold'].tolist()

# 2. Build the novelty reference.
#    It must hold BOTH the known molecules AND the known scaffolds. Generation starts
#    from Bemis-Murcko scaffolds, so a generated structure can reproduce a known
#    scaffold exactly -- and comparing only against whole molecules let those through
#    labelled "novel". In the earlier mTOR run the rank-1 final candidate was a
#    scaffold already present in that target's own scaffold table.
#    InChIKeys are used because SMILES strings compare unreliably.
sar_df = pd.read_csv(SAR_FILE)
original_smiles = set(sar_df['canonical_smiles'].tolist())
known_scaffolds = set(scaffold_df['scaffold'].dropna().tolist())
known_keys = set()
for _smi in list(original_smiles) + list(known_scaffolds):
    _key = pipeline_common.inchikey(_smi)
    if _key:
        known_keys.add(_key)
print(f"✓ Novelty reference: {len(original_smiles)} known molecules + "
      f"{len(known_scaffolds)} known scaffolds -> {len(known_keys)} InChIKeys")

# 3. Convert Seeds to RDKit molecules
seed_mols = []
for smiles in seed_smiles:
    mol = Chem.MolFromSmiles(smiles)
    if mol is not None:
        seed_mols.append((smiles, mol))

print(f"✓ Successfully parsed {len(seed_mols)} seed molecules")

# ============================================================================
# 2. ANALOG GENERATION FUNCTIONS (WITH VALIDATION)
# ============================================================================

def is_valid_molecule(mol, smiles):
    """
    Validate that a molecule is a valid, connected chemical entity.
    Returns: bool: True if valid, False otherwise
    """
    if mol is None: return False

    # Check for disconnected fragments (contains '.')
    if '.' in smiles: return False

    # Check for dummy atoms (contains '*')
    if '*' in smiles: return False

    # Check that molecule has reasonable size
    if mol.GetNumAtoms() < 10 or mol.GetNumAtoms() > 100: return False

    # Check for proper sanitization
    try:
        Chem.SanitizeMol(mol)
        return True
    except:
        return False


def mutate_atoms(mol, n_mutations=30):
    """Generate analogs by mutating atoms to bioisosteric replacements."""
    mutated = []
    # Bioisosteric replacements: (original, replacements)
    replacements = {
        6: [7, 8, 16],      # C -> N, O, S
        7: [6, 8],          # N -> C, O
        8: [6, 7, 16],      # O -> C, N, S
        9: [17, 35],        # F -> Cl, Br
        17: [9, 35],        # Cl -> F, Br
        35: [9, 17],        # Br -> F, Cl
    }

    for _ in range(n_mutations):
        try:
            emol = Chem.RWMol(mol)
            atom_idx = random.randint(0, emol.GetNumAtoms() - 1)
            atom = emol.GetAtomWithIdx(atom_idx)
            atomic_num = atom.GetAtomicNum()

            if atomic_num in replacements and len(replacements[atomic_num]) > 0:
                new_atomic_num = random.choice(replacements[atomic_num])
                atom.SetAtomicNum(new_atomic_num)

                new_mol = emol.GetMol()
                Chem.SanitizeMol(new_mol)
                new_smiles = Chem.MolToSmiles(new_mol)

                if is_valid_molecule(new_mol, new_smiles):
                    mutated.append(new_mol)
        except: continue
    return mutated


def add_functional_groups(mol, n_variants=20):
    """Add common functional groups to the molecule."""
    variants = []
    functional_groups = [
        'C',           # Methyl
        'CC',          # Ethyl
        'C(C)C',       # Isopropyl
        'OC',          # Methoxy
        'F',           # Fluoro
        'Cl',          # Chloro
        'C(F)(F)F',    # Trifluoromethyl
        'C#N',         # Cyano
        'N',           # Amino
    ]

    for _ in range(n_variants):
        try:
            # Create editable copy
            emol = Chem.RWMol(mol)

            # Find atom to attach to
            atom_idx = random.randint(0, emol.GetNumAtoms() - 1)
            atom = emol.GetAtomWithIdx(atom_idx)

            # Skip if atom already has many bonds
            if atom.GetDegree() >= 3: continue

            # Add functional group
            fg_smiles = random.choice(functional_groups)
            fg_mol = Chem.MolFromSmiles(fg_smiles)

            if fg_mol is not None:
                # Simple approach: combine and sanitize
                combined = Chem.CombineMols(emol, fg_mol)
                new_mol = Chem.RWMol(combined)

                # Add bond between molecules
                new_mol.AddBond(atom_idx, emol.GetNumAtoms(), Chem.BondType.SINGLE)

                # Try to sanitize
                final_mol = new_mol.GetMol()
                Chem.SanitizeMol(final_mol)

                # Validate molecule
                final_smiles = Chem.MolToSmiles(final_mol)

                if is_valid_molecule(final_mol, final_smiles):
                    variants.append(final_mol)
        except: continue
    return variants


def modify_ring_systems(mol, n_variants=20):
    """Modify aromatic rings by substitution."""
    variants = []
    for _ in range(n_variants):
        try:
            emol = Chem.RWMol(mol)

            # Find aromatic atoms
            aromatic_atoms = [atom.GetIdx() for atom in emol.GetAtoms() if atom.GetIsAromatic()]

            if len(aromatic_atoms) > 0:
                # Pick random aromatic atom
                atom_idx = random.choice(aromatic_atoms)
                atom = emol.GetAtomWithIdx(atom_idx)

                # Check if we can add substituent
                if atom.GetDegree() < 3:
                    # Add small substituent
                    sub_smiles = random.choice(['F', 'Cl', 'C', 'OC', 'N'])
                    sub_mol = Chem.MolFromSmiles(sub_smiles)

                    if sub_mol is not None:
                        combined = Chem.CombineMols(emol, sub_mol)
                        new_mol = Chem.RWMol(combined)
                        new_mol.AddBond(atom_idx, emol.GetNumAtoms(), Chem.BondType.SINGLE)

                        final_mol = new_mol.GetMol()
                        Chem.SanitizeMol(final_mol)

                        # Validate molecule
                        final_smiles = Chem.MolToSmiles(final_mol)

                        if is_valid_molecule(final_mol, final_smiles):
                            variants.append(final_mol)
        except: continue
    return variants


def add_substituents(mol, n_variants=20):
    """Add various substituents at different positions."""
    variants = []
    substituents = [
        ('C(C)C', 'isopropyl'), ('C(C)(C)C', 't-butyl'), ('c1ccccc1', 'phenyl'),
        ('C(=O)C', 'acetyl'), ('S(=O)(=O)C', 'methylsulfonyl'),
        ('N(C)C', 'dimethylamino'), ('OC(C)C', 'isopropoxy'),
    ]

    for _ in range(n_variants):
        try:
            emol = Chem.RWMol(mol)

            # Find suitable attachment point
            suitable_atoms = []
            for atom in emol.GetAtoms():
                if atom.GetDegree() < 3 and atom.GetAtomicNum() in [6, 7]:
                    suitable_atoms.append(atom.GetIdx())

            if len(suitable_atoms) > 0:
                atom_idx = random.choice(suitable_atoms)
                sub_smiles, _ = random.choice(substituents)
                sub_mol = Chem.MolFromSmiles(sub_smiles)

                if sub_mol is not None:
                    combined = Chem.CombineMols(emol, sub_mol)
                    new_mol = Chem.RWMol(combined)
                    new_mol.AddBond(atom_idx, emol.GetNumAtoms(), Chem.BondType.SINGLE)

                    final_mol = new_mol.GetMol()
                    Chem.SanitizeMol(final_mol)

                    # Validate molecule
                    final_smiles = Chem.MolToSmiles(final_mol)

                    if is_valid_molecule(final_mol, final_smiles):
                        variants.append(final_mol)
        except: continue
    return variants


print("\n[2/6] Generating analogs for each seed molecule...")
print(f"  Target: ≥50 analogs per seed")
print(f"  NOTE: BRICS fragment strategy DISABLED due to disconnection issues")

all_analogs = []
seed_to_analogs = defaultdict(list)
start_time = time.time()

for i, (seed_smiles, seed_mol) in enumerate(seed_mols):
    if (i+1) % 5 == 0: print(f"  Processing Seed {i+1}/{len(seed_mols)}...")
    seed_analogs = []

    # Apply 4 generation strategies
    seed_analogs.extend(mutate_atoms(seed_mol, n_mutations=30)) # Strategy 1: Atom mutations (bioisosteres) - INCREASED
    seed_analogs.extend(add_functional_groups(seed_mol, n_variants=25)) # Strategy 2: Add functional groups - INCREASED
    seed_analogs.extend(modify_ring_systems(seed_mol, n_variants=25)) # Strategy 3: Modify ring systems - INCREASED
    seed_analogs.extend(add_substituents(seed_mol, n_variants=20)) # Strategy 4: Add larger substituents - NEW

    # Convert to SMILES and deduplicate
    analog_smiles = set()
    for analog in seed_analogs:
        if analog is not None:
            try:
                smi = Chem.MolToSmiles(analog)
                # CRITICAL: Validate before adding
                if smi and smi != seed_smiles and is_valid_molecule(analog, smi):
                    analog_smiles.add((smi, seed_smiles))
            except: continue

    seed_to_analogs[seed_smiles] = list(analog_smiles)
    all_analogs.extend(analog_smiles)

    print(f"    Generated {len(analog_smiles)} unique valid analogs")

    # Progress update
    if (i + 1) % 5 == 0:
        elapsed = time.time() - start_time
        print(f"\n  Progress: {i+1}/{len(seed_mols)} seeds processed ({elapsed:.1f}s)")

print(f"✓ Total valid analogs generated: {len(all_analogs)}")

# ============================================================================
# 3. FILTRATION & OPTIMIZATION
# ============================================================================

print("\n[3/6] Filtering and optimizing candidates...")

# Remove duplicates
unique_analogs = list(set([smi for smi, parent in all_analogs]))
print(f"  After deduplication: {len(unique_analogs)} unique molecules")

# Reject anything reproducing a known molecule OR a known scaffold.
novel_analogs = []
n_known_mol = n_known_struct = 0
for smi in unique_analogs:
    if smi in original_smiles or smi in known_scaffolds:
        n_known_mol += 1
        continue
    key = pipeline_common.inchikey(smi)
    if key is not None and key in known_keys:
        n_known_struct += 1
        continue
    novel_analogs.append(smi)
print(f"  After novelty filter: {len(novel_analogs)} novel molecules")
print(f"    rejected {n_known_mol} identical to a known molecule or scaffold (SMILES)")
print(f"    rejected {n_known_struct} identical by InChIKey (the check the old "
      f"code missed)")

# CRITICAL: Apply final validation to remove any disconnected molecules
valid_novel_analogs = []
for smi in novel_analogs:
    mol = Chem.MolFromSmiles(smi)
    if is_valid_molecule(mol, smi):
        valid_novel_analogs.append(smi)

print(f"  After connectivity validation: {len(valid_novel_analogs)} valid connected molecules")

# Convert to molecules and calculate properties
candidates = []
for i, smiles in enumerate(valid_novel_analogs):
    if i % 200 == 0 and i > 0: print(f"    Filtering {i}/{len(valid_novel_analogs)}...")

    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol is None: continue

        # Calculate properties
        mw = Descriptors.MolWt(mol)
        logp = Crippen.MolLogP(mol)
        qed = QED.qed(mol)

        # Apply drug-likeness filters
        # Filters: MW 300-600, LogP < 5.5, QED > 0.5
        if 300 <= mw <= 600 and logp < 5.5 and qed > 0.5:
            parent_smiles = next((parent for smi, parent in all_analogs if smi == smiles), None)
            candidates.append({
                'SMILES': smiles,
                'Parent_SMILES': parent_smiles,
                'MW': mw, 'LogP': logp, 'QED': qed, 'mol': mol
            })
    except: continue

print(f"  After property filters: {len(candidates)} drug-like candidates")
if len(candidates) < 20: print(f"  ⚠ WARNING: Low candidate count. Selecting all.")

# ============================================================================
# 4. CANDIDATE SELECTION
# ============================================================================

print("\n[4/6] Calculating Synthetic Accessibility and ranking...")

# Calculate SA scores for remaining candidates
if not SA_dictionary:
    print("  ⚠ WARNING: Using default score (5.0) because data dictionary is empty.")

success_count = 0
for c in candidates:
    try:
        # Pass the dictionary explicitly
        score = calculate_sa_clean(c['mol'], SA_dictionary)
        c['SA_Score'] = score
        c['Combined_Score'] = c['QED'] / (score / 10.0 + 0.1)
        success_count += 1
    except Exception as e:
        c['SA_Score'] = 5.0
        c['Combined_Score'] = c['QED']

print(f"  ✓ Processed {success_count} candidates.")

# Sort and Select
candidates_sorted = sorted(candidates, key=lambda x: x['Combined_Score'], reverse=True)

# Select top 20 unique candidates (or all if less than 20)
n_select = min(20, len(candidates_sorted))
top_candidates = candidates_sorted[:n_select]

print(f"✓ Selected top {n_select} candidates")
if top_candidates:
    print(f"  Combined score range: {top_candidates[-1]['Combined_Score']:.3f} - {top_candidates[0]['Combined_Score']:.3f}")

print("\n🏆 TOP 5 CANDIDATES (True SA Score):")
print(f"{'Rank':<5} | {'SA Score':<10} | {'QED':<10} | {'Combined':<10}")
print("-" * 45)
for i, c in enumerate(top_candidates[:5]):
    print(f"{i+1:<5} | {c['SA_Score']:<10.2f} | {c['QED']:<10.2f} | {c['Combined_Score']:<10.3f}")

# ============================================================================
# 5. SAVE RESULTS
# ============================================================================

print("\n[5/6] Saving results...")

# Create output DataFrame
output_df = pd.DataFrame([{
    'SMILES': c['SMILES'],
    'Parent_SMILES': c['Parent_SMILES'],
    'MW': round(c['MW'], 2),
    'LogP': round(c['LogP'], 2),
    'QED': round(c['QED'], 3),
    'SA_Score': round(c['SA_Score'], 2),
    'Combined_Score': round(c['Combined_Score'], 3)
} for c in top_candidates])

output_df.to_csv(TOP20_CANDIDATES_FILE, index=False)
print(f"✓ Saved top {len(output_df)} candidates to: {TOP20_CANDIDATES_FILE}")

output_df = pd.DataFrame([{
    'SMILES': c['SMILES'],
    'Parent_SMILES': c['Parent_SMILES'],
    'MW': round(c['MW'], 2),
    'LogP': round(c['LogP'], 2),
    'QED': round(c['QED'], 3),
    'SA_Score': round(c['SA_Score'], 2),
    'Combined_Score': round(c['Combined_Score'], 3)
} for c in candidates_sorted])

output_df.to_csv(CANDIDATES_FILE, index=False)
print(f"✓ Saved all {len(output_df)} candidates to: {CANDIDATES_FILE}")

# Print summary statistics
print("\n" + "="*80)
print("CANDIDATE SUMMARY")
print("="*80)
print(f"Total analogs generated:     {len(all_analogs)}")
print(f"Unique molecules:            {len(unique_analogs)}")
print(f"Novel molecules:             {len(novel_analogs)}")
print(f"Valid connected molecules:   {len(valid_novel_analogs)}")
print(f"Drug-like candidates:        {len(candidates)}")
print(f"Top candidates selected:     {len(top_candidates)}")
print(f"\nProperty ranges for all drug-like candidates:")
print(f"  Molecular Weight:  {output_df['MW'].min():.1f} - {output_df['MW'].max():.1f} Da")
print(f"  LogP:              {output_df['LogP'].min():.2f} - {output_df['LogP'].max():.2f}")
print(f"  QED:               {output_df['QED'].min():.3f} - {output_df['QED'].max():.3f}")
print(f"  SA Score:          {output_df['SA_Score'].min():.2f} - {output_df['SA_Score'].max():.2f}")
print(f"  Combined Score:    {output_df['Combined_Score'].min():.2f} - {output_df['Combined_Score'].max():.2f}")

# ============================================================================
# 6. CHEMICAL SPACE VISUALIZATION (Fixed Deprecation)
# ============================================================================

print("\n[6/6] Generating chemical space visualization...")

def get_fingerprint(mol):
    gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)
    return np.array(gen.GetFingerprint(mol))

# 1. Prepare Data Vectors
seed_fps = [get_fingerprint(mol) for _, mol in seed_mols]
cand_fps = [get_fingerprint(c['mol']) for c in candidates_sorted]

if seed_fps and cand_fps:
    # Stack: [Seeds, Top20, Rest]
    X = np.vstack([seed_fps, cand_fps])

    # Indices for slicing later
    idx_seeds_end = len(seed_fps)
    idx_top20_end = idx_seeds_end + 20

    # --- A. PCA ---
    print("  Calculating PCA...")
    pca = PCA(n_components=2, random_state=42)
    X_pca = pca.fit_transform(X)

    plt.figure(figsize=(12, 10))
    # 1. Plot "Rest" (Background, Grey)
    plt.scatter(X_pca[idx_top20_end:, 0], X_pca[idx_top20_end:, 1],
                c='black', s=30, alpha=0.5, label='Other Candidates')
    # 2. Plot Seeds (Blue)
    plt.scatter(X_pca[:idx_seeds_end, 0], X_pca[:idx_seeds_end, 1],
                c='blue', s=100, alpha=0.8, edgecolors='k', label='Seeds')
    # 3. Plot Top 20 (Red Stars, Top Layer)
    plt.scatter(X_pca[idx_seeds_end:idx_top20_end, 0], X_pca[idx_seeds_end:idx_top20_end, 1],
                c='red', s=200, marker='*', edgecolors='white', linewidth=1.5, label='Top 20 Candidates')

    plt.title(f'Chemical Space (PCA): {TARGET_NAME.upper()}', fontsize=14, fontweight='bold')
    plt.xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)")
    plt.ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)")
    plt.legend(loc='best')
    plt.grid(True, linestyle='--', alpha=0.3)
    plt.savefig(FIGURES_DIR / f"{TARGET_NAME}_generation_pca.png", dpi=300, bbox_inches='tight')
    print("  ✓ Saved generation_pca.png")

    # --- B. t-SNE ---
    # Only run t-SNE if we have enough points, otherwise it looks weird
    if len(X) > 30:
        print("  Calculating t-SNE (this may take a moment)...")
        # Perplexity must be < number of samples. Default 30.
        perp = min(30, len(X) - 1)
        tsne = TSNE(n_components=2, random_state=42, perplexity=perp, n_iter=1000)
        X_tsne = tsne.fit_transform(X)

        plt.figure(figsize=(12, 10))
        # 1. Rest
        plt.scatter(X_tsne[idx_top20_end:, 0], X_tsne[idx_top20_end:, 1],
                    c='black', s=30, alpha=0.5, label='Other Candidates')
        # 2. Seeds
        plt.scatter(X_tsne[:idx_seeds_end, 0], X_tsne[:idx_seeds_end, 1],
                    c='blue', s=100, alpha=0.8, edgecolors='k', label='Seeds')
        # 3. Top 20
        plt.scatter(X_tsne[idx_seeds_end:idx_top20_end, 0], X_tsne[idx_seeds_end:idx_top20_end, 1],
                    c='red', s=200, marker='*', edgecolors='white', linewidth=1.5, label='Top 20 Candidates')

        plt.title(f'Chemical Space (t-SNE): {TARGET_NAME.upper()}', fontsize=14, fontweight='bold')
        plt.xlabel("Dimension 1")
        plt.ylabel("Dimension 2")
        plt.legend(loc='best')
        plt.grid(True, linestyle='--', alpha=0.3)
        plt.savefig(FIGURES_DIR / f"{TARGET_NAME}_generation_tsne.png", dpi=300, bbox_inches='tight')
        print("  ✓ Saved generation_tsne.png")
    else:
        print("  ⚠️ Skipping t-SNE (not enough data points)")


    # --- Grid Image of TOP 20 with FULL SCORES ---
    print("  Generating structure grid with detailed scores...")

    # 1. CRITICAL FIX: Use 'candidates_sorted' here, not 'candidates'
    #    This ensures the structures match the Ranking/CSV.
    top20_list = candidates_sorted[:20]
    top20_mols = [c['mol'] for c in top20_list]

    # 2. Create Labels (Rank | Combined | QED | SA)
    legends = []
    for i, c in enumerate(top20_list):
        label = (f"Rank {i+1}\n"
                 f"Score: {c['Combined_Score']:.2f}\n"
                 f"QED: {c['QED']:.2f} | SA: {c['SA_Score']:.2f}")
        legends.append(label)

    # 3. Draw Grid (Single call, replacing the previous double-call logic)
    img = Draw.MolsToGridImage(top20_mols,
                               molsPerRow=5,
                               subImgSize=(220, 220),
                               legends=legends,
                               returnPNG=False)

    # Save high-res version
    img.save(str(FIGURES_DIR / f"{TARGET_NAME}_top20_structures.png"))
    print("  ✓ Saved top20_structures.png with correct rankings.")

print("\n" + "="*80)
print("ANALOGS GENERATION COMPLETE")
print("="*80)
print(f"Ready for docking in Step 5.")

In [ ]:
#!/usr/bin/env python3
"""
Step 5: Virtual screening by molecular docking (AutoDock Vina).

The binding site is located by pipeline_common.resolve_box_center(), which groups
HETATM records *by residue* and uses one named reference ligand. The previous
implementation averaged the coordinates of every non-water HETATM in the file
into a single point. With one hetero group that happens to be correct; with a
cofactor, an ion, a second ligand copy or a dimer present it silently returns a
point in empty space. Measured on the structures used here:

    EGFR 5Y9T (1 hetero group)   0.0 A from the ligand  -- accidentally correct
    KRAS 7RT1 (2 ligands+GDP+Mg) 7.3 A off              -- pocket clipped
    mTOR 4JSV (ADP x2 in a dimer) 37.7 A off            -- box contained no pocket

No step prints a reassuring tick unless the box demonstrably encloses the pocket.
"""

import os
import subprocess
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import requests
from rdkit import Chem
from rdkit.Chem import AllChem
import warnings
warnings.filterwarnings('ignore')

import pipeline_common
CFG = pipeline_common.get_config(quiet=True)
pipeline_common.export_globals(CFG, globals())

# Docking runs through the Vina executable. The Python bindings have no wheel for
# Python 3.13 (Colab's current interpreter) and fail to build from source there.
VINA_BIN = pipeline_common.find_vina_binary(CFG.get("vina_binary"))
if not VINA_BIN:
    raise RuntimeError(
        "AutoDock Vina executable not found. Re-run Step 0, or install it with:\n"
        "    !apt-get install -y autodock-vina")

CANDIDATES_CSV = RESULTS_DIR / f"{TARGET_NAME}_top20_generated_candidates.csv"
DOCKING_RESULTS_CSV = RESULTS_DIR / f"{TARGET_NAME}_docking_results.csv"
DOCKING_FIGURE = FIGURES_DIR / f"{TARGET_NAME}_docking_scores.png"

print("=" * 80)
print(f"STEP 5: MOLECULAR DOCKING — {TARGET_NAME.upper()}")
print("=" * 80)


# ==============================================================================
# 1. Receptor structure
# ==============================================================================
def fetch_pdb(pdb_id: str) -> Path:
    """Return a local path to the configured PDB, downloading it if needed."""
    path = WORKFLOW_DATA_DIR / f"{pdb_id}.pdb"
    if path.exists():
        print(f"  Using local structure: {path.name}")
        return path
    print(f"  Downloading {pdb_id} from RCSB ...")
    resp = requests.get(f"https://files.rcsb.org/download/{pdb_id}.pdb", timeout=60)
    if resp.status_code != 200:
        raise RuntimeError(
            f"Could not download {pdb_id} (HTTP {resp.status_code}). Either place "
            f"{pdb_id}.pdb in {WORKFLOW_DATA_DIR} by hand, or set a different "
            f"'manual_pdb_id' for target '{TARGET_NAME}' in pipeline_common.py.")
    path.write_text(resp.text, encoding="utf-8")
    print(f"  Saved {path.name}")
    return path


print("\n[1/6] RECEPTOR")
print("-" * 80)

# The structure is named by the config, never guessed from directory order. The old
# code took os.listdir()[0], so a second PDB in the folder silently changed target.
PDB_ID = CFG.get("manual_pdb_id")
if not PDB_ID:
    raise RuntimeError(
        f"No 'manual_pdb_id' set for target '{TARGET_NAME}'. Step 3 reports the "
        f"candidate structures it found; choose one and set it in pipeline_common.py "
        f"so the run is reproducible.")

PDB_FILE = fetch_pdb(PDB_ID)
RECEPTOR_PDBQT = WORKFLOW_DATA_DIR / f"{PDB_ID}_receptor.pdbqt"

# --- locate the pocket (the critical step) ------------------------------------
center, site = pipeline_common.resolve_box_center(PDB_FILE, CFG)
box_size = pipeline_common.fit_box_size(site, CFG)
BOX_SIZE = box_size

# --- build the receptor, keeping structurally integral cofactors --------------
clean_pdb = WORKFLOW_DATA_DIR / f"{PDB_ID}_clean.pdb"
pipeline_common.write_receptor(PDB_FILE, clean_pdb, CFG, exclude_reference=site)

print("  Converting receptor to PDBQT (OpenBabel)...")
result = subprocess.run(
    ['obabel', str(clean_pdb), '-O', str(RECEPTOR_PDBQT), '-xr'],
    capture_output=True, text=True, timeout=300)
if not RECEPTOR_PDBQT.exists() or RECEPTOR_PDBQT.stat().st_size == 0:
    raise RuntimeError(f"Receptor PDBQT not produced. obabel said:\n{result.stderr}")
print(f"  ✓ {RECEPTOR_PDBQT.name}")
print("  NOTE: OpenBabel does not assign histidine protonation states. For work you "
      "intend to\n        publish, prepare the receptor with prepare_receptor4.py "
      "(ADFR) or PDBFixer.")


# ==============================================================================
# 2. Ligand preparation
# ==============================================================================
def prepare_ligand(smiles: str, name: str, out_dir: Path):
    """3D-embed, MMFF-minimise and convert one SMILES to PDBQT."""
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            return None
        mol = Chem.AddHs(mol)

        params = AllChem.ETKDGv3()
        params.randomSeed = int(CFG['random_seed'])   # reproducible geometry
        cids = AllChem.EmbedMultipleConfs(mol, numConfs=int(CFG['n_conformers']),
                                          params=params)
        if not cids:
            if AllChem.EmbedMolecule(mol, randomSeed=int(CFG['random_seed'])) != 0:
                return None
            cids = [0]

        energies = AllChem.MMFFOptimizeMoleculeConfs(mol, maxIters=1000)
        converged = [(e, cid) for cid, (ok, e) in zip(cids, energies) if ok == 0]
        best_cid = int(min(converged)[1]) if converged else int(cids[0])

        pdb_file = out_dir / f"{name}.pdb"
        Chem.MolToPDBFile(mol, str(pdb_file), confId=best_cid)

        pdbqt_file = out_dir / f"{name}.pdbqt"
        subprocess.run(['obabel', str(pdb_file), '-O', str(pdbqt_file), '-p', '7.4'],
                       capture_output=True, text=True, timeout=120)
        if not pdbqt_file.exists():
            return None
        return {'pdbqt': str(pdbqt_file),
                'heavy_atoms': Chem.RemoveHs(mol).GetNumHeavyAtoms()}
    except Exception:
        return None


print("\n[2/6] CANDIDATE LIGANDS")
print("-" * 80)

if not CANDIDATES_CSV.exists():
    raise FileNotFoundError(
        f"{CANDIDATES_CSV} not found — run Step 4 first (it generates the candidates).")

df = pd.read_csv(CANDIDATES_CSV)
print(f"  Loaded {len(df)} candidates from {CANDIDATES_CSV.name}")

print("\n[3/6] PREPARING LIGANDS")
print("-" * 80)

ligands, failed = [], []
for idx, row in df.iterrows():
    name = f"candidate_{idx + 1:02d}"
    prepared = prepare_ligand(row['SMILES'], name, LIGAND_PREP_DIR)
    if prepared:
        ligands.append({'index': idx, 'name': name, 'smiles': row['SMILES'],
                        'original_score': row.get('Combined_Score', 0.0), **prepared})
        print(f"  [{idx + 1:>2}/{len(df)}] {name} ✓ ({prepared['heavy_atoms']} heavy atoms)")
    else:
        failed.append(name)
        print(f"  [{idx + 1:>2}/{len(df)}] {name} ✗ preparation failed")

print(f"\n  Prepared {len(ligands)}/{len(df)}")
if failed:
    print(f"  Failed: {', '.join(failed)}")
if not ligands:
    raise RuntimeError("No ligands could be prepared.")


# ==============================================================================
# 3. Docking
# ==============================================================================
print("\n[4/6] DOCKING")
print("-" * 80)
print(f"  Centre        : ({center[0]:.2f}, {center[1]:.2f}, {center[2]:.2f})")
print(f"  Box           : {box_size:.1f} Å cube")
print(f"  Pocket ref    : {site['reference']} ({site['n_atoms']} atoms, "
      f"source: {site['source']})")
print(f"  Exhaustiveness: {CFG['exhaustiveness']}   poses per ligand: {CFG['num_modes']}")
print(f"  Engine        : {VINA_BIN}")
print("-" * 80)

results = []
for i, lig in enumerate(ligands, 1):
    pose_path = RESULTS_DIR / f"{lig['name']}_docked.pdbqt"
    affinity, poses, err = pipeline_common.dock_with_vina_cli(
        RECEPTOR_PDBQT, lig['pdbqt'], center, box_size, pose_path,
        cfg=CFG, vina_bin=VINA_BIN)
    if affinity is None:
        print(f"  [{i:>2}/{len(ligands)}] {lig['name']} ✗  {err}")
        continue
    le = pipeline_common.ligand_efficiency(affinity, lig['heavy_atoms'])
    pose_range = (max(poses) - min(poses)) if poses and len(poses) > 1 else 0.0
    results.append({
        'Candidate_ID': lig['name'],
        'Original_Index': lig['index'],
        'SMILES': lig['smiles'],
        'Affinity_kcal_mol': round(affinity, 3),
        'Ligand_Efficiency': round(le, 4) if le is not None else None,
        'Heavy_Atoms': lig['heavy_atoms'],
        'Pose_Spread_kcal_mol': round(pose_range, 3),
        'N_Poses': len(poses) if poses else 0,
        'Combined_Score': lig['original_score'],
    })
    print(f"  [{i:>2}/{len(ligands)}] {lig['name']}  {affinity:7.2f} kcal/mol   "
          f"LE {le:.3f}   pose spread {pose_range:.2f}")

if not results:
    raise RuntimeError("Every docking run failed.")


# ==============================================================================
# 4. Ranking — with an explicit check that the spread is worth ranking
# ==============================================================================
print("\n[5/6] RESULTS")
print("-" * 80)

results_df = pd.DataFrame(results).sort_values('Affinity_kcal_mol')
results_df['Rank'] = range(1, len(results_df) + 1)
results_df = results_df[['Rank', 'Candidate_ID', 'SMILES', 'Affinity_kcal_mol',
                         'Ligand_Efficiency', 'Heavy_Atoms',
                         'Pose_Spread_kcal_mol', 'N_Poses', 'Original_Index']]
results_df.to_csv(DOCKING_RESULTS_CSV, index=False)
print(f"  Saved {DOCKING_RESULTS_CSV.name}")

aff = results_df['Affinity_kcal_mol']
spread = aff.max() - aff.min()
vina_error = CFG['vina_score_error_kcal']

print(f"\n  best {aff.min():.2f} | mean {aff.mean():.2f} ± {aff.std():.2f} | "
      f"worst {aff.max():.2f} kcal/mol")
print(f"  spread across candidates: {spread:.2f} kcal/mol")
if not pipeline_common.spread_is_meaningful(aff.tolist(), vina_error, factor=1.0):
    print(f"  !! That spread is smaller than Vina's own scoring error "
          f"(~{vina_error} kcal/mol).")
    print(f"     The ORDER of these candidates is not meaningful. Use the scores to "
          f"reject\n     obvious non-binders, not to pick a winner.")
else:
    print(f"  Spread exceeds Vina's ~{vina_error} kcal/mol error, so coarse "
          f"discrimination is defensible.")

print(f"\n  Ranking by ligand efficiency instead of raw affinity (raw Vina scores "
      f"reward size):")
for _, row in results_df.nlargest(5, 'Ligand_Efficiency').iterrows():
    print(f"    {row['Candidate_ID']}  LE {row['Ligand_Efficiency']:.3f}  "
          f"({row['Affinity_kcal_mol']:.2f} kcal/mol, {row['Heavy_Atoms']} heavy atoms)")

print(f"\n  TOP 5 BY AFFINITY")
for _, row in results_df.head(5).iterrows():
    print(f"    {row['Rank']}. {row['Candidate_ID']}  {row['Affinity_kcal_mol']:.2f} "
          f"kcal/mol   LE {row['Ligand_Efficiency']:.3f}")
    print(f"       {row['SMILES']}")


# ==============================================================================
# 5. Figure
# ==============================================================================
print("\n[6/6] FIGURE")
print("-" * 80)

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.size'] = 9

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

colors = ['#2ca02c' if i < 5 else '#cccccc' for i in range(len(results_df))]
ax1.bar(range(len(results_df)), results_df['Affinity_kcal_mol'],
        color=colors, edgecolor='black', linewidth=0.5)
mean_aff = aff.mean()
ax1.axhline(mean_aff, color='red', linestyle='--', linewidth=1, alpha=0.7)
# Show Vina's error as a band, so the reader can see whether the differences matter.
ax1.axhspan(mean_aff - vina_error / 2, mean_aff + vina_error / 2,
            color='red', alpha=0.10)
ax1.set_xlabel('Candidate (ranked)', fontweight='bold')
ax1.set_ylabel('Binding affinity (kcal/mol)', fontweight='bold')
ax1.set_title(f'Docking affinity — {TARGET_NAME.upper()} vs {PDB_ID}',
              fontweight='bold', pad=12)
ax1.set_xticks(range(len(results_df)))
ax1.set_xticklabels(range(1, len(results_df) + 1), fontsize=8)
ax1.grid(axis='y', alpha=0.3, linestyle='--')
ax1.set_axisbelow(True)
ax1.legend(handles=[
    Patch(facecolor='#2ca02c', edgecolor='black', label='Top 5'),
    Patch(facecolor='#cccccc', edgecolor='black', label='Others'),
    Patch(facecolor='red', alpha=0.10, label=f"Vina error ±{vina_error / 2:.2f}"),
], loc='lower right', frameon=True)

ax2.scatter(results_df['Heavy_Atoms'], results_df['Affinity_kcal_mol'],
            c='steelblue', s=60, edgecolors='black', linewidth=0.5)
ax2.set_xlabel('Heavy atoms', fontweight='bold')
ax2.set_ylabel('Binding affinity (kcal/mol)', fontweight='bold')
ax2.set_title('Affinity vs. size\n(a downward trend means size, not fit)',
              fontweight='bold', pad=12)
ax2.grid(alpha=0.3, linestyle='--')
if len(results_df) > 2:
    r = np.corrcoef(results_df['Heavy_Atoms'], results_df['Affinity_kcal_mol'])[0, 1]
    ax2.text(0.04, 0.05, f'r = {r:.2f}', transform=ax2.transAxes,
             bbox=dict(boxstyle='round', facecolor='white', alpha=0.85))

plt.tight_layout()
plt.savefig(DOCKING_FIGURE, dpi=300, bbox_inches='tight')
plt.close()
print(f"  Saved {DOCKING_FIGURE.name}")

# Record how the site was chosen, so the run can be audited later.
site_record = {k: v for k, v in site.items() if k != 'reference_coords'}
site_record.update({'box_center': list(center), 'box_size': box_size,
                    'pdb_id': PDB_ID, 'exhaustiveness': CFG['exhaustiveness'],
                    'num_modes': CFG['num_modes']})
import json
(RESULTS_DIR / f"{TARGET_NAME}_docking_site.json").write_text(
    json.dumps(pipeline_common.json_safe(site_record), indent=2), encoding='utf-8')

print("\n" + "=" * 80)
print("STEP 5 COMPLETE")
print("=" * 80)
print(f"  {DOCKING_RESULTS_CSV}")
print(f"  {DOCKING_FIGURE}")
print(f"  {RESULTS_DIR / f'{TARGET_NAME}_docking_site.json'}  (how the box was chosen)")


In [ ]:
# ==============================================================================
# 🧬 VISUALIZE TOP 5: FINAL (Smooth Gradient + Simple Text Legend)
# ==============================================================================
import sys
import json
import subprocess
import pandas as pd
from pathlib import Path

# 1. SETUP
try: import py3Dmol
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "py3Dmol", "-q"])
    import py3Dmol

import pipeline_common
CFG = pipeline_common.get_config(quiet=True)
pipeline_common.export_globals(CFG, globals())
DOCKING_RESULTS_CSV = RESULTS_DIR / f"{TARGET_NAME}_docking_results.csv"

# The structure is named by the config, not picked by directory order. On the
# automatic-search path manual_pdb_id is None, so read back the PDB id Step 3
# actually resolved instead of building a path called "None_clean.pdb".
PDB_ID = CFG['manual_pdb_id']
if not PDB_ID:
    _sa = RESULTS_DIR / f"{TARGET_NAME}_structural_analysis.json"
    if not _sa.exists():
        raise RuntimeError(
            "No PDB id available: manual_pdb_id is unset and Step 3 has not run, "
            f"so there is no {_sa.name} to read the resolved structure from.")
    PDB_ID = json.loads(_sa.read_text(encoding="utf-8"))["structure"]["pdb_id"]
    print(f"ℹ️  PDB id taken from Step 3's saved analysis: {PDB_ID}")

clean_pdb_path = WORKFLOW_DATA_DIR / f"{PDB_ID}_clean.pdb"
raw_pdb_path = WORKFLOW_DATA_DIR / f"{PDB_ID}.pdb"

if clean_pdb_path.exists():
    RECEPTOR_PDB = clean_pdb_path
    print(f"✅ Using CLEAN receptor: {RECEPTOR_PDB.name} (Original ligand removed)")
else:
    RECEPTOR_PDB = raw_pdb_path
    print(f"⚠️ Clean receptor not found. Using RAW: {RECEPTOR_PDB.name}")

def convert_pdbqt_to_pdb(pdbqt_path):
    pdb_path = pdbqt_path.with_suffix(".pdb")
    cmd = ['obabel', '-ipdbqt', str(pdbqt_path), '-opdb', '-O', str(pdb_path)]
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode == 0 and pdb_path.exists(): return pdb_path
    return None

# 2. VISUALIZATION LOOP
if not DOCKING_RESULTS_CSV.exists():
    print("❌ Results file missing.")
else:
    df = pd.read_csv(DOCKING_RESULTS_CSV).sort_values('Affinity_kcal_mol', ascending=True)
    top_5 = df.head(5)

    print(f"\n👀 Visualizing Top {len(top_5)} Candidates...\n")

    for rank, (_, row) in enumerate(top_5.iterrows(), start=1):
        cand_id = row['Candidate_ID']
        affinity = row['Affinity_kcal_mol']

        docked_file = RESULTS_DIR / f"{cand_id}_docked.pdbqt"
        if not docked_file.exists(): docked_file = RESULTS_DIR / f"{cand_id}_BEST_docked.pdbqt"
        if not docked_file.exists(): continue

        print(f"🔹 Rank {rank}: {cand_id} (Affinity: {affinity:.2f} kcal/mol)")

        viz_pdb = convert_pdbqt_to_pdb(docked_file)

        if viz_pdb:
            view = py3Dmol.view(width=700, height=500)

            # A. RECEPTOR (Cleaned)
            with open(RECEPTOR_PDB, 'r') as f:
                view.addModel(f.read(), "pdb")

            # SURFACE: SES (PyMOL Style) + Smooth Gradient
            view.addSurface(py3Dmol.SES, {
                'opacity': 0.85,
                'colorscheme': {
                    'prop': 'hydrophobicity',
                    'gradient': 'rwb',   # Red-White-Blue smooth gradient
                    'min': -1.5,
                    'max': 1.5
                }
            }, {'model': 0})

            # B. LIGAND
            with open(viz_pdb, 'r') as f:
                view.addModel(f.read(), "pdb")
            view.setStyle({'model': 1}, {"stick": {'colorscheme': 'greenCarbon', 'radius': 0.2}})

            # C. LABELS
            # Title
            view.addLabel(f"{cand_id} ({affinity:.2f} kcal/mol)",
                          {'position': {'x':10, 'y':10, 'z':0}, 'useScreen': True,
                           'backgroundColor': 'black', 'fontColor': 'white'})

            # Instruction
            view.addLabel("💡 SHIFT + Click & Drag down to Slice View",
                          {'position': {'x':430, 'y':10, 'z':0}, 'useScreen': True,
                           'backgroundColor': '#ffffcc', 'fontColor': 'black', 'border': '1px solid black'})

            # D. SIMPLE TEXT LEGEND (Bottom Right)
            legend_text = "Red: Hydrophobic | White: Neutral | Blue: Hydrophilic"
            view.addLabel(legend_text,
                          {'position': {'x':180, 'y':460, 'z':0}, 'useScreen': True,
                           'fontColor': 'black', 'backgroundColor': 'white', 'fontSize': 12, 'border': '1px solid #ccc'})

            view.zoomTo({'model': 1})
            view.show()
            print("-" * 60)

In [ ]:
#!/usr/bin/env python3
"""
Step 6: Machine-learning potency model (Random Forest on Morgan fingerprints).

Two changes that matter for whether the output means anything:

1. TRAINING RANGE. The model trains on {target}_bioactivity_full.csv — every
   curated compound across the full measured potency range. Training only on the
   sub-50 nM hits truncates the labels into a ~2 log-unit band, and a regressor
   with no weak examples cannot learn what makes a compound weak.

2. APPLICABILITY DOMAIN. Every candidate is scored for nearest-neighbour
   similarity to the training set. A random forest handed an out-of-domain
   molecule does not fail — it returns the training mean, which then looks like a
   confident prediction. In the previous runs the candidate predictions varied 3x
   to 11x less than the model's own RMSE, so ranking them ranked noise.

Both the random split and a scaffold-based split are reported. The scaffold number
is the honest one: random splits put close analogues on both sides.
"""

import json
import os
import sys
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from rdkit import Chem, DataStructs
from rdkit.Chem.Scaffolds import MurckoScaffold
from scipy.stats import pearsonr, spearmanr
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GroupShuffleSplit, cross_val_score, train_test_split
import warnings
warnings.filterwarnings('ignore')

import pipeline_common
CFG = pipeline_common.get_config(quiet=True)
pipeline_common.export_globals(CFG, globals())

plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.size'] = 10

FULL_DATA_FILE = RESULTS_DIR / f"{TARGET_NAME}_bioactivity_full.csv"
HITS_DATA_FILE = RESULTS_DIR / f"{TARGET_NAME}_inhibitors_cleaned.csv"
CANDIDATES_FILE = RESULTS_DIR / f"{TARGET_NAME}_top20_generated_candidates.csv"
DOCKING_RESULTS_FILE = RESULTS_DIR / f"{TARGET_NAME}_docking_results.csv"

# Every output carries the target prefix. Previously the model and two figures did
# not, so running a second target in the same session silently overwrote them — and
# Step 8 would then score one target's candidates with another target's model.
MODEL_FILE = RESULTS_DIR / f"{TARGET_NAME}_pIC50_model.pkl"
PREDICTIONS_FILE = RESULTS_DIR / f"{TARGET_NAME}_candidate_predictions.csv"
METRICS_FILE = RESULTS_DIR / f"{TARGET_NAME}_model_metrics.json"
PERFORMANCE_PLOT = FIGURES_DIR / f"{TARGET_NAME}_model_performance.png"
COMPARISON_PLOT = FIGURES_DIR / f"{TARGET_NAME}_ml_docking_comparison.png"

print("=" * 80)
print(f"STEP 6: MACHINE LEARNING MODEL — {TARGET_NAME.upper()}")
print("=" * 80)

# ==============================================================================
# 1. Training data — full measured range
# ==============================================================================
print("\n[1/7] Training data")
print("-" * 80)

if FULL_DATA_FILE.exists():
    df_train = pd.read_csv(FULL_DATA_FILE)
    print(f"  {FULL_DATA_FILE.name}: {len(df_train)} compounds (full measured range)")
elif HITS_DATA_FILE.exists():
    df_train = pd.read_csv(HITS_DATA_FILE)
    print(f"  !! {FULL_DATA_FILE.name} not found; falling back to "
          f"{HITS_DATA_FILE.name} ({len(df_train)} compounds).")
    print(f"     That file holds only potent hits, so the label range is truncated "
          f"and this model\n     will not be able to rank candidates. Re-run Step 1 "
          f"to generate the full-range file.")
else:
    raise FileNotFoundError(
        f"No training data. Run Step 1 first (expected {FULL_DATA_FILE.name}).")

span = df_train['pIC50'].max() - df_train['pIC50'].min()
print(f"  pIC50 range: {df_train['pIC50'].min():.2f} – {df_train['pIC50'].max():.2f} "
      f"(span {span:.2f}, sd {df_train['pIC50'].std():.2f})")
if span < 3.0:
    print(f"  !! A {span:.2f} log-unit span is narrow. Treat every number below as "
          f"weakly supported.")

# ==============================================================================
# 2. Features
# ==============================================================================
print("\n[2/7] Morgan fingerprints")
print("-" * 80)

fps, keep = [], []
for i, smi in enumerate(df_train['canonical_smiles']):
    fp = pipeline_common.morgan_fp(smi, CFG)
    if fp is not None:
        arr = np.zeros((CFG['fp_bits'],), dtype=np.int8)
        DataStructs.ConvertToNumpyArray(fp, arr)
        fps.append(arr)
        keep.append(i)

df_train = df_train.iloc[keep].reset_index(drop=True)
X = np.vstack(fps)
y = df_train['pIC50'].values
print(f"  X {X.shape}  y {y.shape}  (radius {CFG['fp_radius']}, {CFG['fp_bits']} bits)")

# Bemis-Murcko scaffold per compound, used for the honest split.
scaffolds = []
for smi in df_train['canonical_smiles']:
    try:
        mol = Chem.MolFromSmiles(smi)
        scaffolds.append(Chem.MolToSmiles(MurckoScaffold.GetScaffoldForMol(mol))
                         if mol else "")
    except Exception:
        scaffolds.append("")
print(f"  {len(set(scaffolds))} distinct scaffolds across {len(df_train)} compounds")

# ==============================================================================
# 3. Train and evaluate
# ==============================================================================
print("\n[3/7] Training and evaluation")
print("-" * 80)

seed = int(CFG['random_seed'])


def fit_rf():
    return RandomForestRegressor(n_estimators=int(CFG['n_estimators']),
                                 random_state=seed, n_jobs=-1)


# --- random split (comparable with the earlier runs) -------------------------
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=CFG['test_size'], random_state=seed)
model = fit_rf()
model.fit(X_tr, y_tr)
pred_te = model.predict(X_te)
rand = {
    'r2': float(r2_score(y_te, pred_te)),
    'rmse': float(np.sqrt(mean_squared_error(y_te, pred_te))),
    'mae': float(mean_absolute_error(y_te, pred_te)),
    'n_test': int(len(y_te)),
}
print(f"  random split      R2 {rand['r2']:+.3f}   RMSE {rand['rmse']:.3f}   "
      f"MAE {rand['mae']:.3f}   (n={rand['n_test']})")

# --- scaffold splits, REPEATED (no shared scaffolds between train and test) --
#
# A single GroupShuffleSplit draw is one sample from a wide distribution. On the
# first live KRAS run it returned R2 = 0.794 against a random-split 0.614 -- the
# wrong way round, since holding out whole scaffolds is normally the harder test.
# That was the draw talking. Repeating it and reporting mean +/- sd shows how much
# of any single number is the split choice rather than the model.
scaf = None
n_repeats = int(CFG['scaffold_split_repeats'])
if len(set(scaffolds)) > 4:
    print(f"  scaffold splits   running {n_repeats} repeats ...")
    splitter = GroupShuffleSplit(n_splits=n_repeats, test_size=CFG['test_size'],
                                 random_state=seed)
    s_r2, s_rmse, s_n = [], [], []
    for k, (s_tr, s_te) in enumerate(splitter.split(X, y, groups=scaffolds), 1):
        s_model = fit_rf()
        s_model.fit(X[s_tr], y[s_tr])
        s_pred = s_model.predict(X[s_te])
        s_r2.append(float(r2_score(y[s_te], s_pred)))
        s_rmse.append(float(np.sqrt(mean_squared_error(y[s_te], s_pred))))
        s_n.append(int(len(s_te)))

    r2_sum = pipeline_common.summarise_scores(s_r2)
    rmse_sum = pipeline_common.summarise_scores(s_rmse)
    stable = pipeline_common.estimate_is_stable(r2_sum)

    # 'r2' / 'rmse' stay as the MEAN so every downstream reader keeps working.
    scaf = {
        'r2': r2_sum['mean'],
        'rmse': rmse_sum['mean'],
        'r2_sd': r2_sum['sd'],
        'rmse_sd': rmse_sum['sd'],
        'r2_min': r2_sum['min'],
        'r2_max': r2_sum['max'],
        'r2_per_split': r2_sum['values'],
        'n_repeats': r2_sum['n'],
        'n_test': int(round(sum(s_n) / len(s_n))),
        'stable': bool(stable),
    }
    print(f"  scaffold split    R2 {scaf['r2']:+.3f} +/- {scaf['r2_sd']:.3f}   "
          f"RMSE {scaf['rmse']:.3f} +/- {scaf['rmse_sd']:.3f}   "
          f"({scaf['n_repeats']} repeats, mean n={scaf['n_test']})")
    print(f"                    R2 range {scaf['r2_min']:+.3f} to "
          f"{scaf['r2_max']:+.3f}  <- the honest estimate is the mean, not the best")
    if not stable:
        print("  !! Those repeats disagree enough that no single scaffold-split "
              "number is quotable.")
        print("     Report the mean with its spread, never one draw.")
        print("     A wide spread usually means too few distinct scaffolds for "
              "the held-out set")
        print("     to be representative of the chemistry.")
    if scaf['r2'] > rand['r2']:
        print("  Note: mean scaffold-split R2 still exceeds the random-split R2.")
        print("        That is unusual - holding out whole scaffolds should be "
              "the harder test.")
        print("        Worth checking whether the scaffold column is as diverse "
              "as the")
        print("        compound count suggests.")

cv = cross_val_score(fit_rf(), X_tr, y_tr, cv=int(CFG['cv_folds']),
                     scoring='r2', n_jobs=-1)
print(f"  {CFG['cv_folds']}-fold CV on train  R2 {cv.mean():+.3f} ± {cv.std():.3f}")

# A baseline that ignores the molecule entirely. If the model cannot beat it, the
# fingerprints carry no usable signal for this dataset.
baseline_rmse = float(np.sqrt(mean_squared_error(y_te, np.full_like(y_te, y_tr.mean()))))
print(f"  mean-only baseline RMSE {baseline_rmse:.3f}  "
      f"-> model is {100 * (1 - rand['rmse'] / baseline_rmse):+.0f}% better")
if rand['rmse'] >= baseline_rmse:
    print(f"  !! The model is no better than predicting the mean for every compound.")

joblib.dump(model, MODEL_FILE)
print(f"  Model saved: {MODEL_FILE.name}")

# ==============================================================================
# 4. Candidates: predict, then check whether the prediction is usable
# ==============================================================================
print("\n[4/7] Candidate predictions")
print("-" * 80)

if not CANDIDATES_FILE.exists():
    raise FileNotFoundError(
        f"{CANDIDATES_FILE.name} not found — run Step 4 first. (The old code printed "
        f"'skipping' here and then crashed with a NameError on the next line.)")

df_cand = pd.read_csv(CANDIDATES_FILE)
cand_fps, cand_keep = [], []
for i, smi in enumerate(df_cand['SMILES']):
    fp = pipeline_common.morgan_fp(smi, CFG)
    if fp is not None:
        arr = np.zeros((CFG['fp_bits'],), dtype=np.int8)
        DataStructs.ConvertToNumpyArray(fp, arr)
        cand_fps.append(arr)
        cand_keep.append(i)
    else:
        print(f"  !! unparseable candidate {i}: {smi}")

df_cand = df_cand.iloc[cand_keep].reset_index(drop=True)
X_cand = np.vstack(cand_fps)
df_cand['Predicted_pIC50'] = model.predict(X_cand)
df_cand['Predicted_IC50_nM'] = 10 ** (9.0 - df_cand['Predicted_pIC50'])

# --- applicability domain ----------------------------------------------------
ad = pipeline_common.max_tanimoto_to_training(
    df_cand['SMILES'].tolist(), df_train['canonical_smiles'].tolist(), CFG)
df_cand['AD_Max_Tanimoto'] = [round(v, 3) if v is not None else None for v in ad]
threshold = CFG['applicability_min_tanimoto']
df_cand['In_Domain'] = [bool(v is not None and v >= threshold) for v in ad]

n_out = int((~df_cand['In_Domain']).sum())
print(f"  Predicted {len(df_cand)} candidates")
print(f"  Nearest-neighbour Tanimoto to training set: "
      f"{df_cand['AD_Max_Tanimoto'].min():.2f} – {df_cand['AD_Max_Tanimoto'].max():.2f}")
print(f"  Outside the applicability domain (< {threshold}): {n_out}/{len(df_cand)}")
if n_out:
    print(f"  !! Those {n_out} predictions are extrapolation. The model returns "
          f"something close to the\n     training mean for molecules it has no "
          f"neighbours for; that is not a potency estimate.")

# --- is the prediction spread worth ranking? --------------------------------
pred_sd = float(df_cand['Predicted_pIC50'].std())
pred_span = float(df_cand['Predicted_pIC50'].max() - df_cand['Predicted_pIC50'].min())
model_error = scaf['rmse'] if scaf else rand['rmse']
usable = pipeline_common.spread_is_meaningful(
    df_cand['Predicted_pIC50'].tolist(), model_error, factor=2.0)

print(f"\n  prediction spread {pred_span:.3f} pIC50 (sd {pred_sd:.3f})")
print(f"  model error       {model_error:.3f} RMSE")
if usable:
    print(f"  Spread is at least 2x the model error, so the ranking carries signal.")
else:
    print(f"  !! The spread is SMALLER than the model's own error. Ranking these "
          f"candidates by\n     predicted potency ranks noise. Step 7 will therefore "
          f"drop the potency term\n     from the consensus score rather than let it "
          f"decide the outcome.")
    print(f"     Report these as 'predicted active, not ranked', and treat the "
          f"ordering as arbitrary.")
    print(f"     Fix properly by training on a wider potency range, or by generating "
          f"candidates\n     closer to the training chemistry (Step 4 starts from "
          f"bare scaffolds).")

df_cand['Potency_Rankable'] = bool(usable)
df_cand.to_csv(PREDICTIONS_FILE, index=False)
print(f"\n  Saved {PREDICTIONS_FILE.name}")

metrics = {
    'target': TARGET_NAME,
    'model': 'RandomForestRegressor',
    'n_estimators': int(CFG['n_estimators']),
    'features': f"Morgan r={CFG['fp_radius']} {CFG['fp_bits']} bits",
    'n_train_compounds': int(len(df_train)),
    'training_pic50_range': [float(df_train['pIC50'].min()), float(df_train['pIC50'].max())],
    'random_split': rand,
    'scaffold_split': scaf,
    'cv_r2_mean': float(cv.mean()),
    'cv_r2_std': float(cv.std()),
    'mean_only_baseline_rmse': baseline_rmse,
    'candidate_prediction_sd': pred_sd,
    'candidate_prediction_span': pred_span,
    'n_candidates_out_of_domain': n_out,
    'potency_rankable': bool(usable),
}
METRICS_FILE.write_text(json.dumps(pipeline_common.json_safe(metrics), indent=2),
                        encoding='utf-8')
print(f"  Saved {METRICS_FILE.name}")

# ==============================================================================
# 5. Performance figure
# ==============================================================================
print("\n[5/7] Performance figure")
print("-" * 80)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

ax1.scatter(y_te, pred_te, alpha=0.6, s=35, c='steelblue', edgecolors='black',
            linewidth=0.4)
lims = [min(y.min(), pred_te.min()) - 0.3, max(y.max(), pred_te.max()) + 0.3]
ax1.plot(lims, lims, 'r--', linewidth=1.2, label='perfect prediction')
ax1.set_xlim(lims)
ax1.set_ylim(lims)
ax1.set_xlabel('Measured pIC50', fontweight='bold')
ax1.set_ylabel('Predicted pIC50', fontweight='bold')
ax1.set_title(f'{TARGET_NAME.upper()} test set (random split)', fontweight='bold')
ax1.grid(alpha=0.3, linestyle='--')
txt = (f"R² = {rand['r2']:.3f}\nRMSE = {rand['rmse']:.3f}\n"
       f"MAE = {rand['mae']:.3f}\nn = {rand['n_test']}")
if scaf:
    txt += (f"\n\nscaffold split ({scaf['n_repeats']}x):"
            f"\nR² = {scaf['r2']:.3f} ± {scaf['r2_sd']:.3f}"
            f"\nRMSE = {scaf['rmse']:.3f}")
ax1.text(0.03, 0.97, txt, transform=ax1.transAxes, va='top', fontsize=9,
         bbox=dict(boxstyle='round', facecolor='white', alpha=0.9))
ax1.legend(loc='lower right')

# Candidate predictions against the training distribution: when the candidates pile
# up at the training mean, the model is extrapolating, not predicting.
ax2.hist(y, bins=35, color='lightsteelblue', edgecolor='black', alpha=0.85,
         label=f'training labels (n={len(y)})')
ax2.axvline(float(np.mean(y)), color='black', linestyle=':', linewidth=1.6,
            label=f'training mean {np.mean(y):.2f}')
for i, v in enumerate(df_cand['Predicted_pIC50']):
    ax2.axvline(v, color='crimson', alpha=0.55, linewidth=1.1,
                label='candidate predictions' if i == 0 else None)
ax2.set_xlabel('pIC50', fontweight='bold')
ax2.set_ylabel('Frequency', fontweight='bold')
ax2.set_title(f'Candidates vs. training distribution\n'
              f'spread {pred_span:.2f} vs model RMSE {model_error:.2f}',
              fontweight='bold')
ax2.grid(axis='y', alpha=0.3, linestyle='--')
ax2.legend(fontsize=8)

plt.tight_layout()
plt.savefig(PERFORMANCE_PLOT, dpi=300, bbox_inches='tight')
plt.close()
print(f"  Saved {PERFORMANCE_PLOT.name}")

# ==============================================================================
# 6. ML vs docking
# ==============================================================================
print("\n[6/7] ML vs docking agreement")
print("-" * 80)

if DOCKING_RESULTS_FILE.exists():
    dock = pd.read_csv(DOCKING_RESULTS_FILE)
    merged = pd.merge(df_cand, dock[['SMILES', 'Affinity_kcal_mol']], on='SMILES',
                      how='inner').drop_duplicates(subset=['SMILES'])
    if len(merged) > 2:
        pr, pp = pearsonr(merged['Predicted_pIC50'], merged['Affinity_kcal_mol'])
        sr, sp = spearmanr(merged['Predicted_pIC50'], merged['Affinity_kcal_mol'])
        print(f"  n = {len(merged)}")
        print(f"  Pearson  r = {pr:+.3f} (p = {pp:.3f})")
        print(f"  Spearman ρ = {sr:+.3f} (p = {sp:.3f})")
        print(f"  Expected sign is NEGATIVE (higher pIC50 ↔ more negative affinity).")
        if pp > 0.05:
            print(f"  !! p = {pp:.3f}: no significant agreement. With n = {len(merged)} "
                  f"this test has very little\n     power, so this is 'no evidence "
                  f"either way', not 'the methods disagree'.")

        fig, ax = plt.subplots(figsize=(8, 6))
        in_dom = merged['In_Domain'].astype(bool)
        ax.scatter(merged.loc[in_dom, 'Affinity_kcal_mol'],
                   merged.loc[in_dom, 'Predicted_pIC50'],
                   s=70, c='steelblue', edgecolors='black', label='in domain')
        ax.scatter(merged.loc[~in_dom, 'Affinity_kcal_mol'],
                   merged.loc[~in_dom, 'Predicted_pIC50'],
                   s=70, c='lightgrey', edgecolors='black', marker='^',
                   label='outside domain')
        ax.set_xlabel('Docking affinity (kcal/mol)', fontweight='bold')
        ax.set_ylabel('ML predicted pIC50', fontweight='bold')
        ax.set_title(f'{TARGET_NAME.upper()}: ML vs docking', fontweight='bold')
        ax.grid(alpha=0.3, linestyle='--')
        ax.text(0.03, 0.97, f'Pearson r = {pr:+.3f}\nSpearman ρ = {sr:+.3f}\n'
                            f'n = {len(merged)}',
                transform=ax.transAxes, va='top', fontsize=9,
                bbox=dict(boxstyle='round', facecolor='white', alpha=0.9))
        ax.legend(loc='lower right', fontsize=9)
        plt.tight_layout()
        plt.savefig(COMPARISON_PLOT, dpi=300, bbox_inches='tight')
        plt.close()
        print(f"  Saved {COMPARISON_PLOT.name}")
    else:
        print("  Too few overlapping candidates for a correlation.")
else:
    print(f"  {DOCKING_RESULTS_FILE.name} not found — run Step 5 for this comparison.")

# ==============================================================================
# 7. Summary
# ==============================================================================
print("\n[7/7] Summary")
print("-" * 80)
print(f"  trained on          : {len(df_train)} compounds, pIC50 "
      f"{df_train['pIC50'].min():.2f}–{df_train['pIC50'].max():.2f}")
print(f"  random-split R²     : {rand['r2']:+.3f}")
if scaf:
    print(f"  scaffold-split R²   : {scaf['r2']:+.3f} ± {scaf['r2_sd']:.3f} "
          f"over {scaf['n_repeats']} repeats "
          f"({'stable' if scaf['stable'] else 'UNSTABLE — quote the spread'})")
print(f"  candidates in domain: {len(df_cand) - n_out}/{len(df_cand)}")
print(f"  potency rankable    : {'yes' if usable else 'NO — spread is below model error'}")

print("\n" + "=" * 80)
print("STEP 6 COMPLETE")
print("=" * 80)


In [ ]:
#!/usr/bin/env python3
"""
Step 7: In silico ADME/Tox Prediction & Final Selection (Generalized)
========================================================
This script performs comprehensive ADME/Tox profiling and selects the best candidates
using Multi-Parameter Optimization (MPO).

Objectives:
- Calculate Lipinski's Rule of 5 and Veber's parameters
- Screen for PAINS (Pan Assay Interference Compounds)
- Implement consensus scoring combining ML predictions, docking, and drug-likeness
- Select top 5 candidates and visualize top 5 with radar plots

Features:
- Target-Agnostic: Works for any target defined in the workflow.
- Dynamic Paths: Auto-detects input/output files based on TARGET_NAME.
"""

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Circle, RegularPolygon
from matplotlib.path import Path
from matplotlib.projections.polar import PolarAxes
from matplotlib.projections import register_projection
from matplotlib.spines import Spine
from matplotlib.transforms import Affine2D
import warnings
from pathlib import Path
warnings.filterwarnings('ignore')

# Import RDKit
try:
    from rdkit import Chem
    from rdkit.Chem import Descriptors, Lipinski, Crippen, rdMolDescriptors
    from rdkit.Chem.FilterCatalog import FilterCatalog, FilterCatalogParams
    print("✓ RDKit imported successfully")
except ImportError as e:
    print(f"ERROR: RDKit not available: {e}")
    print("Installing RDKit...")
    import subprocess
    subprocess.run(["pip", "install", "rdkit", "-q"], check=True)
    from rdkit import Chem
    from rdkit.Chem import Descriptors, Lipinski, Crippen, rdMolDescriptors
    from rdkit.Chem.FilterCatalog import FilterCatalog, FilterCatalogParams
    print("✓ RDKit installed and imported")

# Set random seed for reproducibility
np.random.seed(42)

# Set matplotlib style
plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['font.size'] = 10
plt.rcParams['axes.linewidth'] = 0.5

# ==============================================================================
# CONFIGURATION — do not edit here; see pipeline_common.py
# ==============================================================================
import pipeline_common
CFG = pipeline_common.get_config(quiet=True)
pipeline_common.export_globals(CFG, globals())

print("="*80)
print(f"STEP 7: PHYSICOCHEMICAL PROFILING, STRUCTURAL ALERTS & FINAL SELECTION "
      f"({TARGET_NAME.upper()})")
print("="*80)
print("NOTE: this step computes physicochemical descriptors (MW, LogP, TPSA, QED,")
print("      rotatable bonds, FractionCSP3), Lipinski/Veber rule compliance and PAINS")
print("      substructure alerts. It does NOT predict absorption, distribution,")
print("      metabolism, excretion or toxicity -- there is no BBB, Caco-2, CYP, hERG")
print("      or hepatotoxicity model here. For real ADMET endpoints use ADMETlab,")
print("      pkCSM or SwissADME on the final candidates.")

# Input Files (Dynamic)
PREDICTIONS_FILE = RESULTS_DIR / f"{TARGET_NAME}_candidate_predictions.csv"
DOCKING_RESULTS_FILE = RESULTS_DIR / f"{TARGET_NAME}_docking_results.csv"

# Output Files
ADMET_ANALYSIS_FILE = RESULTS_DIR / f"{TARGET_NAME}_admet_analysis.csv"
FINAL_CANDIDATES_FILE = RESULTS_DIR / f"{TARGET_NAME}_final_candidates.csv"
RADAR_PLOT_FILE = FIGURES_DIR / f"{TARGET_NAME}_candidate_radar_plot.png"

# ============================================================================
# 1. DATA LOADING
# ============================================================================
print("\n[1/6] Loading Data...")

# Load ML predictions
if not PREDICTIONS_FILE.exists():
    print(f"❌ Error: Predictions file missing: {PREDICTIONS_FILE}")
    import sys; sys.exit(1)

pred_df = pd.read_csv(PREDICTIONS_FILE)
print(f"✓ Loaded {len(pred_df)} candidates with ML predictions")
print(f"  Columns: {list(pred_df.columns)}")

# Load docking results
if not DOCKING_RESULTS_FILE.exists():
    print(f"❌ Error: Docking results file missing: {DOCKING_RESULTS_FILE}")
    import sys; sys.exit(1)

dock_df = pd.read_csv(DOCKING_RESULTS_FILE)
print(f"✓ Loaded {len(dock_df)} candidates with docking scores")
print(f"  Columns: {list(dock_df.columns)}")

# Merge datasets on SMILES
# Note: Docking results might use 'Candidate_ID' or 'SMILES' as key.
# We'll try to merge on SMILES first as it's chemically unique.
print("\nMerging datasets on SMILES...")
merged_df = pd.merge(pred_df, dock_df[['SMILES', 'Affinity_kcal_mol', 'Candidate_ID']],
                     on='SMILES', how='inner')

# Deduplicate if necessary (sometimes docking produces multiple poses per ligand).
#
# reset_index matters: the ADME and PAINS blocks below build their own DataFrames
# with a fresh 0..n-1 index and attach them column-wise. Pandas aligns those
# assignments on the index, so the gapped index drop_duplicates leaves behind
# whenever it removes a row would write NaN into MW/LogP/QED/Has_PAINS for every
# row past the gap -- which then reads as a silent Lipinski failure, or raises
# "cannot convert float NaN to integer" in the top-5 printout.
_n_before = len(merged_df)
merged_df = merged_df.drop_duplicates(subset=['SMILES']).reset_index(drop=True)
if len(merged_df) < _n_before:
    print(f"  Dropped {_n_before - len(merged_df)} duplicate SMILES")
if merged_df.empty:
    raise RuntimeError(
        "No candidate survived the merge of ML predictions with docking results: "
        "the two steps share no SMILES string at all, which means one of them "
        "canonicalised differently. Compare the SMILES columns of "
        f"{PREDICTIONS_FILE.name} and {DOCKING_RESULTS_FILE.name}.")
print(f"✓ Merged dataset: {merged_df.shape[0]} candidates with complete data")

# ============================================================================
# 2. ADME PROFILING
# ============================================================================
print("\n[2/6] Computing ADME Properties...")

def calculate_adme_properties(smiles):
    """Calculate comprehensive ADME properties for a molecule."""
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return {
            'MW': None, 'LogP': None, 'HBD': None, 'HBA': None,
            'RotBonds': None, 'TPSA': None, 'QED': None,
            'NumAromaticRings': None, 'FractionCSP3': None
        }

    return {
        'MW': Descriptors.MolWt(mol),
        'LogP': Crippen.MolLogP(mol),
        'HBD': Lipinski.NumHDonors(mol),
        'HBA': Lipinski.NumHAcceptors(mol),
        'RotBonds': Lipinski.NumRotatableBonds(mol),
        'TPSA': Descriptors.TPSA(mol),
        'QED': Descriptors.qed(mol),
        'NumAromaticRings': rdMolDescriptors.CalcNumAromaticRings(mol),
        'FractionCSP3': rdMolDescriptors.CalcFractionCSP3(mol)
    }

# Calculate ADME properties for all candidates
print("Calculating ADME descriptors...")
adme_data = []
for idx, row in merged_df.iterrows():
    if (idx + 1) % 5 == 0:
        print(f"  Processing: {idx + 1}/{len(merged_df)}")

    props = calculate_adme_properties(row['SMILES'])
    adme_data.append(props)

# Add ADME properties to dataframe
adme_df = pd.DataFrame(adme_data)
for col in adme_df.columns:
    if col not in merged_df.columns or col == 'QED':  # Recalculate QED for consistency
        merged_df[col] = adme_df[col]

print(f"✓ ADME properties calculated for {len(merged_df)} candidates")

# ============================================================================
# 3. PAINS FILTERING
# ============================================================================
print("\n[3/6] Screening for PAINS (Pan Assay Interference Compounds)...")

# Initialize PAINS filter
params = FilterCatalogParams()
params.AddCatalog(FilterCatalogParams.FilterCatalogs.PAINS)
catalog = FilterCatalog(params)

def check_pains(smiles):
    """Check if molecule contains PAINS substructures."""
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return True, "Invalid SMILES"

    matches = catalog.GetMatches(mol)
    if matches:
        return True, "; ".join([match.GetDescription() for match in matches])
    return False, "No PAINS"

# Check PAINS for all candidates
print("Checking PAINS alerts...")
pains_results = []
for idx, smiles in enumerate(merged_df['SMILES']):
    if (idx + 1) % 5 == 0:
        print(f"  Processing: {idx + 1}/{len(merged_df)}")

    has_pains, description = check_pains(smiles)
    pains_results.append({'Has_PAINS': has_pains, 'PAINS_Description': description})

pains_df = pd.DataFrame(pains_results)
merged_df['Has_PAINS'] = pains_df['Has_PAINS']
merged_df['PAINS_Description'] = pains_df['PAINS_Description']

n_pains = merged_df['Has_PAINS'].sum()
print(f"✓ PAINS screening complete: {n_pains}/{len(merged_df)} candidates flagged")

# ============================================================================
# 4. LIPINSKI'S RULE OF 5 & VEBER'S RULES
# ============================================================================
print("\n[4/6] Evaluating Drug-Likeness Rules...")

# Lipinski's Rule of 5
merged_df['Lipinski_Pass'] = (
    (merged_df['MW'] <= 500) &
    (merged_df['LogP'] <= 5) &
    (merged_df['HBD'] <= 5) &
    (merged_df['HBA'] <= 10)
)

# Veber's Rules
merged_df['Veber_Pass'] = (
    (merged_df['RotBonds'] <= 10) &
    (merged_df['TPSA'] <= 140)
)

# Combined drug-likeness
merged_df['DrugLike'] = merged_df['Lipinski_Pass'] & merged_df['Veber_Pass']

n_lipinski = merged_df['Lipinski_Pass'].sum()
n_veber = merged_df['Veber_Pass'].sum()
n_druglike = merged_df['DrugLike'].sum()

print(f"✓ Lipinski's Rule of 5: {n_lipinski}/{len(merged_df)} pass")
print(f"✓ Veber's Rules: {n_veber}/{len(merged_df)} pass")
print(f"✓ Overall Drug-Likeness: {n_druglike}/{len(merged_df)} pass")

# ============================================================================
# 5. MULTI-PARAMETER OPTIMIZATION (MPO) SCORING
# ============================================================================
print("\n[5/6] Computing Consensus Score (Multi-Parameter Optimization)...")

# ---------------------------------------------------------------------------
# Normalisation uses FIXED, physically meaningful bounds from the shared config.
#
# The previous version rescaled each component across only the 20 candidates in the
# batch, so the best always scored 1.0 and the worst 0.0 however small the real
# difference was. Measured on the earlier runs, the mTOR potency spread was 0.52
# pIC50 against a model RMSE of 0.45, and the docking spread 0.97 kcal/mol against
# Vina's ~2.85 kcal/mol error -- both pure noise, both stretched to the full range.
# Fixed bounds also make a consensus score comparable between runs.
# ---------------------------------------------------------------------------
print("Normalizing scoring components (fixed bounds)...")

PIC50_LO, PIC50_HI = CFG['score_bounds']['pic50']
AFF_LO, AFF_HI = CFG['score_bounds']['affinity']

print(f"  > potency  bounds: pIC50 {PIC50_LO} to {PIC50_HI}  "
      f"(observed {merged_df['Predicted_pIC50'].min():.2f} to "
      f"{merged_df['Predicted_pIC50'].max():.2f})")
print(f"  > docking  bounds: {AFF_LO} to {AFF_HI} kcal/mol  "
      f"(observed {merged_df['Affinity_kcal_mol'].min():.2f} to "
      f"{merged_df['Affinity_kcal_mol'].max():.2f})")

merged_df['Score_Potency'] = pipeline_common.normalize_fixed(
    merged_df['Predicted_pIC50'], PIC50_LO, PIC50_HI)
merged_df['Score_Docking'] = pipeline_common.normalize_fixed(
    merged_df['Affinity_kcal_mol'], AFF_LO, AFF_HI, reverse=True)
merged_df['Score_QED'] = merged_df['QED']
merged_df['Score_PAINS'] = (~merged_df['Has_PAINS']).astype(float)
merged_df['Score_DrugLike'] = merged_df['DrugLike'].astype(float)

# ---------------------------------------------------------------------------
# Drop a component whose spread is below its own measurement error, instead of
# letting noise decide the ranking. Step 6 records whether its predictions are
# rankable at all; docking is checked against Vina's published error here.
# ---------------------------------------------------------------------------
weights = dict(CFG['weights'])
dropped = []

potency_rankable = True
if 'Potency_Rankable' in merged_df.columns:
    potency_rankable = bool(merged_df['Potency_Rankable'].iloc[0])
elif not pipeline_common.spread_is_meaningful(
        merged_df['Predicted_pIC50'].tolist(), 0.5, factor=2.0):
    potency_rankable = False
if not potency_rankable:
    dropped.append('Potency')
    print("  !! Predicted potency varies by less than the model's own error, so it "
          "cannot order\n     these candidates. Dropping the potency term from the "
          "consensus score.")

if not pipeline_common.spread_is_meaningful(
        merged_df['Affinity_kcal_mol'].tolist(),
        CFG['vina_score_error_kcal'], factor=1.0):
    dropped.append('Docking')
    print(f"  !! Docking scores span less than Vina's ~"
          f"{CFG['vina_score_error_kcal']} kcal/mol error, so they cannot order\n"
          f"     these candidates either. Dropping the docking term.")

if 'In_Domain' in merged_df.columns:
    n_out = int((~merged_df['In_Domain'].astype(bool)).sum())
    if n_out:
        print(f"  !! {n_out}/{len(merged_df)} candidates are outside the model's "
              f"applicability domain;\n     their predicted potency is "
              f"extrapolation, not an estimate.")

for key in dropped:
    weights[key] = 0.0
total = sum(weights.values())
if total <= 0:
    raise RuntimeError(
        "Every quantitative term was dropped as noise -- there is nothing left to "
        "rank on. Report these candidates as an unordered set, and fix the upstream "
        "cause (wider training range in Step 1, candidates closer to the training "
        "chemistry in Step 4).")
weights = {k: v / total for k, v in weights.items()}   # renormalise to sum to 1
print(f"  > weights in use: "
      + ", ".join(f"{k} {v:.2f}" for k, v in weights.items() if v > 0))
if dropped:
    print(f"  > dropped as noise: {', '.join(dropped)}")

merged_df['Scoring_Dropped_Terms'] = ';'.join(dropped) if dropped else ''

_unused_weights = None   # weights now come from CFG['weights'] above

merged_df['Consensus_Score'] = (
    weights['Potency'] * merged_df['Score_Potency'] +
    weights['Docking'] * merged_df['Score_Docking'] +
    weights['QED'] * merged_df['Score_QED'] +
    weights['PAINS'] * merged_df['Score_PAINS'] +
    weights['DrugLike'] * merged_df['Score_DrugLike']
)

print("✓ Consensus scores calculated")
print(f"  Score range: {merged_df['Consensus_Score'].min():.3f} - {merged_df['Consensus_Score'].max():.3f}")
print(f"  Mean score: {merged_df['Consensus_Score'].mean():.3f}")

# Rank candidates by consensus score
merged_df = merged_df.sort_values('Consensus_Score', ascending=False).reset_index(drop=True)
merged_df['Consensus_Rank'] = range(1, len(merged_df) + 1)

# ============================================================================
# 6. FINAL SELECTION
# ============================================================================
print("\n[6/6] Selecting Top Candidates...")

# Select top 5 candidates
top5_df = merged_df.head(5).copy()

print("\n" + "="*80)
print("TOP 5 DRUG CANDIDATES (Ranked by Consensus Score)")
print("="*80)
for idx, row in top5_df.iterrows():
    print(f"\nRank {row['Consensus_Rank']}: {row['Candidate_ID']}")
    print(f"  SMILES: {row['SMILES']}")
    print(f"  Consensus Score: {row['Consensus_Score']:.3f}")
    print(f"  Predicted pIC50: {row['Predicted_pIC50']:.2f} (IC50 = {row['Predicted_IC50_nM']:.2f} nM)")
    print(f"  Docking Affinity: {row['Affinity_kcal_mol']:.2f} kcal/mol")
    print(f"  QED: {row['QED']:.3f}")
    print(f"  MW: {row['MW']:.1f} | LogP: {row['LogP']:.2f} | HBD: {int(row['HBD'])} | HBA: {int(row['HBA'])}")
    print(f"  TPSA: {row['TPSA']:.1f} | RotBonds: {int(row['RotBonds'])}")
    print(f"  PAINS: {'⚠ FLAGGED' if row['Has_PAINS'] else '✓ Clean'}")
    print(f"  Drug-like: {'✓ Yes' if row['DrugLike'] else '✗ No'}")

# ============================================================================
# 7. VISUALIZATION: RADAR PLOT FOR TOP 5
# ============================================================================
print("\n" + "="*80)
print("Generating Radar Plot for Top 5 Candidates...")
print("="*80)

# Prepare data for radar plot
top5_df = merged_df.head(5)

# Define properties for radar plot (all normalized 0-1)
properties = ['Potency\n(pIC50)', 'Docking\nAffinity', 'QED\n(Drug-like)',
              'Lipinski\nCompliance', 'Veber\nCompliance', 'PAINS\nClean']

# Create radar data
radar_data = []
for _, row in top5_df.iterrows():
    values = [
        row['Score_Potency'],
        row['Score_Docking'],
        row['Score_QED'],
        1.0 if row['Lipinski_Pass'] else 0.0,
        1.0 if row['Veber_Pass'] else 0.0,
        row['Score_PAINS']
    ]
    radar_data.append(values)

print("\n" + "="*50)
print("RADAR PLOT DATA VERIFICATION")
print("="*50)
properties_list = ['Potency', 'Docking', 'QED', 'Lipinski', 'Veber', 'PAINS']

for i, (values, (_, row)) in enumerate(zip(radar_data, top5_df.iterrows())):
    cand_id = row.get('Candidate_ID', f'Candidate_{i+1}')
    print(f"\n{cand_id}:")
    for prop, score in zip(properties_list, values):
        print(f"  - {prop:<10}: {score:.4f}")
print("="*50 + "\n")

# Create radar plot
fig, ax = plt.subplots(figsize=(10, 8), subplot_kw=dict(projection='polar'))

# Number of variables
num_vars = len(properties)
angles = np.linspace(0, 2 * np.pi, num_vars, endpoint=False).tolist()
angles += angles[:1]  # Complete the circle

# Colors for each candidate
colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd']
labels = [f"Rank {i+1}: {row['Candidate_ID']}\nScore: {row['Consensus_Score']:.3f}"
          for i, (_, row) in enumerate(top5_df.iterrows())]

# Plot each candidate
for idx, (data, color, label) in enumerate(zip(radar_data, colors, labels)):
    data += data[:1]  # Complete the circle
    ax.plot(angles, data, 'o-', linewidth=2, color=color, label=label)
    ax.fill(angles, data, alpha=0.15, color=color)

# Customize plot
ax.set_theta_offset(np.pi / 2)
ax.set_theta_direction(-1)
ax.set_xticks(angles[:-1])
ax.set_xticklabels(properties, size=10)
ax.set_ylim(0, 1)
ax.set_yticks([0.2, 0.4, 0.6, 0.8, 1.0])
ax.set_yticklabels(['0.2', '0.4', '0.6', '0.8', '1.0'], size=8)
ax.set_rlabel_position(180 / num_vars)
ax.grid(True, linestyle='--', linewidth=0.5, alpha=0.7)

# Add legend
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1), fontsize=9)

# Title
plt.title(f'{TARGET_NAME.upper()}: Multi-Parameter Profile of Top 5 Candidates\n(Normalized 0-1 Scale)',
          size=14, weight='bold', pad=20)

# Save figure
plt.tight_layout()
plt.savefig(RADAR_PLOT_FILE, dpi=300, bbox_inches='tight')
plt.close()

print(f"✓ Radar plot saved: {RADAR_PLOT_FILE}")

# ============================================================================
# 8. SAVE RESULTS
# ============================================================================
print("\n" + "="*80)
print("Saving Results...")
print("="*80)

# Save full ADME/Tox analysis
output_cols = [
    'Consensus_Rank', 'Candidate_ID', 'SMILES', 'Consensus_Score',
    'Predicted_pIC50', 'Predicted_IC50_nM', 'Affinity_kcal_mol',
    'MW', 'LogP', 'HBD', 'HBA', 'RotBonds', 'TPSA', 'QED',
    'NumAromaticRings', 'FractionCSP3',
    'Lipinski_Pass', 'Veber_Pass', 'DrugLike', 'Has_PAINS', 'PAINS_Description',
    'Score_Potency', 'Score_Docking', 'Score_QED', 'Score_PAINS', 'Score_DrugLike'
]

merged_df[output_cols].to_csv(ADMET_ANALYSIS_FILE, index=False)
print(f"✓ Saved: {ADMET_ANALYSIS_FILE}")

# Save top 5 final candidates
final_cols = [
    'Consensus_Rank', 'Candidate_ID', 'SMILES', 'Consensus_Score',
    'Predicted_pIC50', 'Predicted_IC50_nM', 'Affinity_kcal_mol',
    'MW', 'LogP', 'HBD', 'HBA', 'RotBonds', 'TPSA', 'QED',
    'Lipinski_Pass', 'Veber_Pass', 'DrugLike', 'Has_PAINS'
]

top5_df[final_cols].to_csv(FINAL_CANDIDATES_FILE, index=False)
print(f"✓ Saved: {FINAL_CANDIDATES_FILE}")

# ============================================================================
# 9. SUMMARY STATISTICS
# ============================================================================
print("\n" + "="*80)
print("ADME/TOX ANALYSIS SUMMARY")
print("="*80)
print(f"\nTotal Candidates Analyzed: {len(merged_df)}")
print(f"\nDrug-Likeness Compliance:")
print(f"  Lipinski's Rule of 5: {n_lipinski}/{len(merged_df)} ({100*n_lipinski/len(merged_df):.1f}%)")
print(f"  Veber's Rules: {n_veber}/{len(merged_df)} ({100*n_veber/len(merged_df):.1f}%)")
print(f"  Combined Drug-Like: {n_druglike}/{len(merged_df)} ({100*n_druglike/len(merged_df):.1f}%)")
print(f"\nPAINS Screening:")
print(f"  Clean Candidates: {len(merged_df) - n_pains}/{len(merged_df)} ({100*(len(merged_df)-n_pains)/len(merged_df):.1f}%)")
print(f"  Flagged Candidates: {n_pains}/{len(merged_df)} ({100*n_pains/len(merged_df):.1f}%)")

print(f"\nConsensus Score Distribution:")
print(f"  Range: {merged_df['Consensus_Score'].min():.3f} - {merged_df['Consensus_Score'].max():.3f}")
print(f"  Mean: {merged_df['Consensus_Score'].mean():.3f} ± {merged_df['Consensus_Score'].std():.3f}")
print(f"  Median: {merged_df['Consensus_Score'].median():.3f}")

if not top5_df.empty:
    print(f"\nTop Candidate ({top5_df.iloc[0]['Candidate_ID']}):")
    print(f"  Consensus Score: {top5_df.iloc[0]['Consensus_Score']:.3f}")
    print(f"  Predicted IC50: {top5_df.iloc[0]['Predicted_IC50_nM']:.2f} nM")
    print(f"  Docking Affinity: {top5_df.iloc[0]['Affinity_kcal_mol']:.2f} kcal/mol")
    print(f"  QED: {top5_df.iloc[0]['QED']:.3f}")
    print(f"  Drug-Like: {'Yes' if top5_df.iloc[0]['DrugLike'] else 'No'}")
    print(f"  PAINS: {'Flagged' if top5_df.iloc[0]['Has_PAINS'] else 'Clean'}")

print("\n" + "="*80)
print("ADME/TOX ANALYSIS COMPLETE!")
print("="*80)
print("\nOutputs:")
print(f"  1. {ADMET_ANALYSIS_FILE} - Full ADME/Tox analysis")
print(f"  2. {FINAL_CANDIDATES_FILE} - Top 5 candidates")
print(f"  3. {RADAR_PLOT_FILE} - Multi-parameter visualization")
print("\n✓ Step 7 Complete: Ready for final reporting and experimental validation")

In [ ]:
#!/usr/bin/env python3
"""
Final Project Consolidation & Handoff Preparation (Generalized)
Aggregates metrics from all 7 computational steps for writing agent handoff
"""

import json
import pandas as pd
import numpy as np
from pathlib import Path
import sys
import os

# ==============================================================================
# CONFIGURATION — do not edit here; see pipeline_common.py
# ==============================================================================
import pipeline_common
CFG = pipeline_common.get_config(quiet=True)
pipeline_common.export_globals(CFG, globals())

print("=" * 80)
print(f"FINAL PROJECT CONSOLIDATION ({TARGET_NAME.upper()}) - Starting")
print("=" * 80)

# Paths from the shared config
BASE_PATH = BASE_DIR
RESULTS_PATH = RESULTS_DIR
FIGURES_PATH = FIGURES_DIR
WORKFLOW_DATA_PATH = WORKFLOW_DATA_DIR

# Initialize metrics dictionary
project_metrics = {
    "project_name": f"{TARGET_NAME.upper()} Inhibitor Discovery via Computational Drug Design",
    "completion_date": pipeline_common.today(),   # was hard-coded, so all three runs
                                                  # claimed the same completion date
    "total_steps": 7,
    "target": TARGET_NAME,
    "steps": {}
}

print("\n[1/7] Extracting Step 1: Data Acquisition metrics...")
try:
    # Read cleaned bioactivity data (checking both potential locations)
    file_name = f"{TARGET_NAME}_inhibitors_cleaned.csv"
    file_path = RESULTS_PATH / file_name

    if file_path.exists():
        data = pd.read_csv(file_path)

        step1_metrics = {
            "step_name": "Data Acquisition",
            "description": f"Retrieved {TARGET_NAME.upper()} inhibitor data from ChEMBL database",
            "total_inhibitors": len(data),
            "pic50_min": float(data['pIC50'].min()),
            "pic50_max": float(data['pIC50'].max()),
            "pic50_mean": float(data['pIC50'].mean()),
            "pic50_median": float(data['pIC50'].median()),
            "data_source": "ChEMBL Database",
            "output_file": str(file_path.relative_to(BASE_PATH))
        }
        project_metrics["steps"]["step_1_data_acquisition"] = step1_metrics
        print(f"   ✓ Total inhibitors: {step1_metrics['total_inhibitors']}")
        print(f"   ✓ pIC50 range: {step1_metrics['pic50_min']:.2f} - {step1_metrics['pic50_max']:.2f}")
    else:
        raise FileNotFoundError(f"Could not find inhibitors data for {TARGET_NAME}")

except Exception as e:
    print(f"   ✗ Error in Step 1: {e}")
    step1_metrics = {"error": str(e)}
    project_metrics["steps"]["step_1_data_acquisition"] = step1_metrics

print("\n[2/7] Extracting Step 2: SAR Analysis metrics...")
try:
    # Read SAR analysis results
    sar_file = RESULTS_PATH / f"{TARGET_NAME}_sar_analysis.csv"
    scaffold_file = RESULTS_PATH / f"{TARGET_NAME}_scaffold_analysis.csv"

    if sar_file.exists() and scaffold_file.exists():
        sar_data = pd.read_csv(sar_file)
        scaffold_data = pd.read_csv(scaffold_file)

        # Most potent scaffold overall (often a singleton: one measurement).
        top_scaffold = scaffold_data.nlargest(1, 'mean_pIC50')
        # Most potent scaffold that actually recurs -- the defensible headline number.
        recurring = (scaffold_data[scaffold_data['count'] >= int(CFG['min_scaffold_count'])]
                     .nlargest(1, 'mean_pIC50'))

        step2_metrics = {
            "step_name": "Structure-Activity Relationship Analysis",
            "description": "Analyzed privileged scaffolds and SAR patterns",
            "total_molecules_analyzed": len(sar_data),
            # len(), not count.sum(): summing the per-scaffold compound counts gives
            # the number of COMPOUNDS, which made every compound look structurally
            # unique (229/229, 1322/1322, 2128/2128) and so implied no SAR at all.
            "unique_scaffolds": int(len(scaffold_data)),
            "total_compounds_in_scaffold_table": int(scaffold_data['count'].sum()),
            "scaffolds_above_min_count": int(
                (scaffold_data['count'] >= int(CFG['min_scaffold_count'])).sum()),
            "min_scaffold_count": int(CFG['min_scaffold_count']),
            "top_scaffold": {
                "smiles": str(top_scaffold['scaffold'].values[0]) if len(top_scaffold) > 0 else "N/A",
                "count": int(top_scaffold['count'].values[0]) if len(top_scaffold) > 0 else 0,
                "mean_pic50": float(top_scaffold['mean_pIC50'].values[0]) if len(top_scaffold) > 0 else None,
                "std_pic50": float(top_scaffold['std_pIC50'].values[0]) if len(top_scaffold) > 0 else None,
                "note": ("mean of a single measurement; standard deviation undefined"
                         if len(top_scaffold) > 0
                         and int(top_scaffold['count'].values[0]) < 2 else None),
            },
            "top_recurring_scaffold": (
                {
                    "smiles": str(recurring.iloc[0]['scaffold']),
                    "count": int(recurring.iloc[0]['count']),
                    "mean_pic50": float(recurring.iloc[0]['mean_pIC50']),
                }
                if len(recurring) else None),
            "output_files": [
                f"results/{TARGET_NAME}_sar_analysis.csv",
                f"results/{TARGET_NAME}_scaffold_analysis.csv",
                f"figures/{TARGET_NAME}_chemical_space_pca.png",
                f"figures/{TARGET_NAME}_physicochemical_properties.png",
                f"figures/{TARGET_NAME}_top_scaffolds_potency.png"
            ]
        }
        project_metrics["steps"]["step_2_sar_analysis"] = step2_metrics
        print(f"   ✓ Molecules analyzed: {step2_metrics['total_molecules_analyzed']}")
        print(f"   ✓ Top scaffold mean pIC50: {step2_metrics['top_scaffold']['mean_pic50']:.2f}")
    else:
        print(f"   ✗ Step 2 files missing for {TARGET_NAME}")

except Exception as e:
    print(f"   ✗ Error in Step 2: {e}")
    step2_metrics = {"error": str(e)}
    project_metrics["steps"]["step_2_sar_analysis"] = step2_metrics

# ... (Previous parts of the script) ...

print("\n[3/7] Extracting Step 3: Structural Analysis metrics...")

# --- HELPER: Parse PDB Header for Resolution ---
def get_pdb_resolution_from_file(pdb_path):
    """Reads the PDB header to find the resolution."""
    try:
        with open(pdb_path, 'r') as f:
            for i, line in enumerate(f):
                if i > 100: break # Header usually at top
                if "REMARK   2 RESOLUTION" in line:
                    # Typical line: REMARK   2 RESOLUTION.    1.90 ANGSTROMS.
                    parts = line.split()
                    for part in parts:
                        try:
                            # Return the first float found
                            return float(part)
                        except ValueError:
                            continue
    except Exception:
        return "N/A"
    return "N/A"

try:
    # 1. Try reading JSON first
    target_json_path = RESULTS_PATH / f"{TARGET_NAME}_structural_analysis.json"
    if not target_json_path.exists():
        target_json_path = RESULTS_PATH / "structural_analysis.json"

    # Initialize defaults
    pdb_id = "N/A"
    resolution = "N/A"
    ligand_info = {}
    binding_site = []

    # Load JSON if exists
    if target_json_path.exists():
        with open(target_json_path, "r") as f:
            target_data = json.load(f)
            pdb_id = target_data.get("pdb_id", "N/A")
            resolution = target_data.get("resolution", "N/A")
            ligand_info = target_data.get("ligand", {})
            binding_site = target_data.get("binding_site_residues", [])

    # 2. ROBUST FALLBACK: If JSON missed the ID/Resolution, check the raw files
    if pdb_id == "N/A" or resolution == "N/A":
        # Look for PDB files in workflow data
        workflow_data = BASE_PATH / "workflow" / "data"
        pdb_files = list(workflow_data.glob("*.pdb"))

        # Filter out _clean files to find the original
        raw_pdbs = [f for f in pdb_files if "_clean" not in f.stem]

        if raw_pdbs:
            # Found a raw PDB file!
            found_pdb = raw_pdbs[0]

            # Update PDB ID if missing
            if pdb_id == "N/A":
                pdb_id = found_pdb.stem

            # Update Resolution if missing (Parse the file header)
            if resolution == "N/A":
                resolution = get_pdb_resolution_from_file(found_pdb)

    step3_metrics = {
        "step_name": "Target Analysis",
        "description": f"Analyzed {TARGET_NAME.upper()} protein structure and binding site",
        "pdb_id": pdb_id,
        "resolution": resolution,
        "ligand_info": ligand_info,
        "binding_site_residues": binding_site,
        "output_files": [
            f"results/{target_json_path.name}" if target_json_path.exists() else "N/A",
            "results/literature_findings.txt"
        ]
    }
    project_metrics["steps"]["step_3_structural_analysis"] = step3_metrics

    print(f"   ✓ PDB ID: {step3_metrics['pdb_id']}")
    print(f"   ✓ Resolution: {step3_metrics['resolution']}")

except Exception as e:
    print(f"   ✗ Error in Step 3: {e}")
    step3_metrics = {"error": str(e)}
    project_metrics["steps"]["step_3_structural_analysis"] = step3_metrics

print("\n[4/7] Extracting Step 4: Analogs Generation metrics...")
try:
    # Read generated candidates
    gen_file = RESULTS_PATH / f"{TARGET_NAME}_generated_candidates.csv"

    if gen_file.exists():
        gen_data = pd.read_csv(gen_file)

        step4_metrics = {
            "step_name": "Analogs Generation",
            "description": f"Generated novel {TARGET_NAME.upper()} inhibitor candidates using scaffold decoration",
            "total_candidates_generated": len(gen_data),
            "valid_molecules": len(gen_data[gen_data['Valid'] == True]) if 'Valid' in gen_data.columns else len(gen_data),
            "generation_method": "Scaffold-based decoration with functional group enumeration",
            "source_scaffold": step2_metrics.get("top_scaffold", {}).get("smiles", "N/A") if 'step2_metrics' in locals() else "N/A",
            "output_files": [
                f"results/{TARGET_NAME}_generated_candidates.csv",
                f"results/{TARGET_NAME}_top20_generated_candidates.csv"
                f"figures/{TARGET_NAME}_generation_pca.png",
                f"figures/{TARGET_NAME}_generation_tsne.png"
            ]
        }
        project_metrics["steps"]["step_4_analogs_generation"] = step4_metrics
        print(f"   ✓ Total candidates generated: {step4_metrics['total_candidates_generated']}")
        print(f"   ✓ Valid molecules: {step4_metrics['valid_molecules']}")
    else:
        print(f"   ✗ Generated candidates file missing: {gen_file}")

except Exception as e:
    print(f"   ✗ Error in Step 4: {e}")
    step4_metrics = {"error": str(e)}
    project_metrics["steps"]["step_4_analogs_generation"] = step4_metrics

print("\n[5/7] Extracting Step 5: Virtual Screening (Docking) metrics...")
try:
    # Read docking results
    dock_file = RESULTS_PATH / f"{TARGET_NAME}_docking_results.csv"

    if dock_file.exists():
        docking_data = pd.read_csv(dock_file)

        step5_metrics = {
            "step_name": "Virtual Screening (Molecular Docking)",
            "description": "Evaluated binding affinity of generated candidates using AutoDock Vina",
            "molecules_docked": len(docking_data),
            "best_docking_score": float(docking_data['Affinity_kcal_mol'].min()),
            "mean_docking_score": float(docking_data['Affinity_kcal_mol'].mean()),
            "docking_score_range": {
                "min": float(docking_data['Affinity_kcal_mol'].min()),
                "max": float(docking_data['Affinity_kcal_mol'].max())
            },
            "docking_method": "AutoDock Vina",
            "target_pdb": step3_metrics.get("pdb_id", "N/A") if 'step3_metrics' in locals() else "N/A",
            "output_files": [
                f"results/{TARGET_NAME}_docking_results.csv",
                f"figures/{TARGET_NAME}_docking_scores.png"
            ]
        }
        project_metrics["steps"]["step_5_virtual_screening"] = step5_metrics
        print(f"   ✓ Molecules docked: {step5_metrics['molecules_docked']}")
        print(f"   ✓ Best docking score: {step5_metrics['best_docking_score']:.2f} kcal/mol")
    else:
        print(f"   ✗ Docking results missing: {dock_file}")

except Exception as e:
    print(f"   ✗ Error in Step 5: {e}")
    step5_metrics = {"error": str(e)}
    project_metrics["steps"]["step_5_virtual_screening"] = step5_metrics

# ==============================================================================
# 🛠️ HELPER FUNCTION: CALCULATE REAL ML METRICS
# (Insert this BEFORE the Step 6 extraction block)
# ==============================================================================
import joblib
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split, cross_val_score
from rdkit import Chem
from rdkit.Chem import AllChem, rdFingerprintGenerator

def get_real_ml_metrics(target_name):
    """
    Reloads data and model to calculate ACTUAL performance metrics.
    """
    # Step 6 writes its own metrics file. Reading it avoids recomputing the numbers
    # here with a second train/test split that may not match Step 6's.
    metrics_file = RESULTS_PATH / f"{target_name}_model_metrics.json"
    if metrics_file.exists():
        print(f"   ...reading metrics from {metrics_file.name}")
        saved = json.loads(metrics_file.read_text(encoding="utf-8"))
        rnd = saved.get("random_split") or {}
        scf = saved.get("scaffold_split") or {}
        return {
            "r2_score": rnd.get("r2"),
            "rmse": rnd.get("rmse"),
            "scaffold_split_r2": scf.get("r2"),            # mean over repeats
            "scaffold_split_rmse": scf.get("rmse"),
            "scaffold_split_r2_sd": scf.get("r2_sd"),
            "scaffold_split_repeats": scf.get("n_repeats"),
            "scaffold_split_stable": scf.get("stable"),
            "cross_validation": (
                f"{saved.get('cv_r2_mean')} +/- {saved.get('cv_r2_std')}"),
            "mean_only_baseline_rmse": saved.get("mean_only_baseline_rmse"),
            "candidate_prediction_span": saved.get("candidate_prediction_span"),
            "n_candidates_out_of_domain": saved.get("n_candidates_out_of_domain"),
            "potency_rankable": saved.get("potency_rankable"),
            "training_pic50_range": saved.get("training_pic50_range"),
        }

    print("   ...Recalculating actual ML metrics from saved model...")
    try:
        # 1. Load Data -- full measured range preferred, hits file as a fallback
        data_path = RESULTS_PATH / f"{target_name}_bioactivity_full.csv"
        if not data_path.exists():
             data_path = RESULTS_PATH / f"{target_name}_inhibitors_cleaned.csv"

        if not data_path.exists():
            print(f"   ⚠️ Training data not found for metrics calculation.")
            return None

        df = pd.read_csv(data_path)

        # 2. Generate Features (Same as Step 6)
        mfgen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)
        fps = []
        valid_indices = []
        for idx, smiles in enumerate(df['canonical_smiles']):
            mol = Chem.MolFromSmiles(smiles)
            if mol:
                fps.append(mfgen.GetFingerprint(mol))
                valid_indices.append(idx)

        X = np.array(fps)
        y = df.iloc[valid_indices]['pIC50'].values

        # 3. Load Model
        model_path = RESULTS_PATH / f"{target_name}_pIC50_model.pkl"
        if not model_path.exists():
            print(f"   ⚠️ Model file not found: {model_path}")
            return None

        model = joblib.load(model_path)

        # 4. Re-create Split (Seed 42) & Evaluate
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
        y_pred = model.predict(X_test)

        r2 = r2_score(y_test, y_pred)
        rmse = np.sqrt(mean_squared_error(y_test, y_pred))

        # 5. Quick Cross-Validation (5-fold on training data)
        cv_scores = cross_val_score(model, X_train, y_train, cv=5, scoring='r2', n_jobs=-1)
        cv_mean = cv_scores.mean()

        return {
            "r2_score": float(r2),
            "rmse": float(rmse),
            "cross_validation": f"5-fold CV R²: {cv_mean:.3f} (±{cv_scores.std():.3f})"
        }
    except Exception as e:
        print(f"   ⚠️ Could not calculate real metrics: {e}")
        return None

print("\n[6/7] Extracting Step 6: ML Modeling metrics...")
try:
    # Read candidate predictions
    pred_file = RESULTS_PATH / f"{TARGET_NAME}_candidate_predictions.csv"

    if pred_file.exists():
        pred_data = pd.read_csv(pred_file)

        # --- NEW: GET ACTUAL METRICS ---
        real_metrics = get_real_ml_metrics(TARGET_NAME)

        # Fallback if calculation failed
        if real_metrics is None:
            real_metrics = {
                "r2_score": 0.0,
                "rmse": 0.0,
                "cross_validation": "Calculation Failed"
            }

        # Extract model performance from the data or README
        # Typical R² and RMSE values from previous execution
        step6_metrics = {
            "step_name": "Machine Learning Modeling",
            "description": "Built predictive model for pIC50 using molecular descriptors",
            "model_type": "Random Forest Regressor",
            "features_used": "Morgan Fingerprints (2048-bit)",
            "molecules_predicted": len(pred_data),
            "model_performance": real_metrics,
            "prediction_range": {
                "min_predicted_pic50": float(pred_data['Predicted_pIC50'].min()) if 'Predicted_pIC50' in pred_data.columns else 0.0,
                "max_predicted_pic50": float(pred_data['Predicted_pIC50'].max()) if 'Predicted_pIC50' in pred_data.columns else 0.0
            },
            "output_files": [
                f"results/{TARGET_NAME}_candidate_predictions.csv",
                f"results/{TARGET_NAME}_pIC50_model.pkl",
                f"results/{TARGET_NAME}_model_metrics.json",
                f"figures/{TARGET_NAME}_model_performance.png",
                f"figures/{TARGET_NAME}_ml_docking_comparison.png"
            ]
        }
        project_metrics["steps"]["step_6_ml_modeling"] = step6_metrics
        print(f"   ✓ Model type: {step6_metrics['model_type']}")
        _perf = step6_metrics.get('model_performance') or {}
        for _label, _key in (("R² (random split)", 'r2_score'),
                             ("RMSE", 'rmse'),
                             ("R² (scaffold split)", 'scaffold_split_r2')):
            _val = _perf.get(_key)
            print(f"   ✓ {_label}: "
                  + (f"{_val:.3f}" if isinstance(_val, (int, float)) else "n/a"))
        if _perf.get('scaffold_split_stable') is False:
            print("   !! Scaffold-split R2 varies too much across repeats to quote "
                  "a single value; use the mean and its sd.")
        if _perf.get('potency_rankable') is False:
            print("   !! Predicted potency spread is below the model error: the "
                  "candidate ORDER is not meaningful.")
    else:
        print(f"   ✗ Predictions file missing: {pred_file}")

except Exception as e:
    print(f"   ✗ Error in Step 6: {e}")
    step6_metrics = {"error": str(e)}
    project_metrics["steps"]["step_6_ml_modeling"] = step6_metrics

print("\n[7/7] Extracting Step 7: ADME/Tox & Final Selection metrics...")
try:
    # Read final candidates
    final_file = RESULTS_PATH / f"{TARGET_NAME}_final_candidates.csv"
    admet_file = RESULTS_PATH / f"{TARGET_NAME}_admet_analysis.csv"

    if final_file.exists() and admet_file.exists():
        final_data = pd.read_csv(final_file)
        admet_data = pd.read_csv(admet_file)

        # Extract top 5 candidates
        top_5 = final_data.nlargest(5, 'Consensus_Score')

        top_candidates = []
        for idx, row in top_5.iterrows():
            candidate = {
                "candidate_id": str(row['Candidate_ID']) if 'Candidate_ID' in row else f"Candidate_{idx}",
                "smiles": str(row['SMILES']),
                "consensus_score": float(row['Consensus_Score']),
                "predicted_ic50_nm": float(row['Predicted_IC50_nM']) if 'Predicted_IC50_nM' in row else 0.0,
                "docking_score_kcal_mol": float(row['Affinity_kcal_mol']) if 'Affinity_kcal_mol' in row else 0.0,
                "drug_likeness_qed": float(row['QED']) if 'QED' in row else 0.0,
                "lipinski_pass": bool(row['Lipinski_Pass']) if 'Lipinski_Pass' in row else True,
                "pains_clean": not bool(row['Has_PAINS']) if 'Has_PAINS' in row else True
            }
            top_candidates.append(candidate)

        step7_metrics = {
            "step_name": "ADME/Tox Prediction & Final Selection",
            "description": "In silico ADME/Tox profiling and multi-parameter optimization",
            "total_candidates_evaluated": len(admet_data),
            "final_candidates_selected": len(final_data),
            "selection_criteria": [
                "Predicted pIC50 (30% weight)",
                "Docking Affinity (30% weight)",
                "Drug-likeness QED (20% weight)",
                "PAINS screening (10% weight)",
                "Lipinski/Veber rules (10% weight)"
            ],
            "top_5_candidates": top_candidates,
            "output_files": [
                f"results/{TARGET_NAME}_admet_analysis.csv",
                f"results/{TARGET_NAME}_final_candidates.csv",
                f"figures/{TARGET_NAME}_candidate_radar_plot.png"
            ]
        }
        project_metrics["steps"]["step_7_admet_selection"] = step7_metrics
        print(f"   ✓ Total candidates evaluated: {step7_metrics['total_candidates_evaluated']}")
        print(f"   ✓ Final candidates selected: {step7_metrics['final_candidates_selected']}")
        if top_candidates:
            print(f"   ✓ Top candidate IC50: {top_candidates[0]['predicted_ic50_nm']:.2f} nM")
    else:
        print(f"   ✗ Final candidate files missing for {TARGET_NAME}")

except Exception as e:
    print(f"   ✗ Error in Step 7: {e}")
    step7_metrics = {"error": str(e)}
    project_metrics["steps"]["step_7_admet_selection"] = step7_metrics

# Add summary statistics
print("\n" + "=" * 80)
print("GENERATING SUMMARY STATISTICS")
print("=" * 80)

project_metrics["summary"] = {
    # Filled in after the artifact check below, as the share of the critical
    # artifacts that actually exist. It used to be the literal string "100%",
    # printed even when a step had raised and recorded nothing.
    "pipeline_completeness": None,
    "total_inhibitors_screened": step1_metrics.get("total_inhibitors", 0) if 'step1_metrics' in locals() else 0,
    "novel_candidates_generated": step4_metrics.get("total_candidates_generated", 0) if 'step4_metrics' in locals() else 0,
    "final_leads_identified": step7_metrics.get("final_candidates_selected", 0) if 'step7_metrics' in locals() else 0,
    "best_predicted_ic50_nm": min([c['predicted_ic50_nm'] for c in step7_metrics.get("top_5_candidates", [])]) if 'step7_metrics' in locals() and step7_metrics.get("top_5_candidates") else 0.0,
    "best_docking_score": step5_metrics.get("best_docking_score", 0.0) if 'step5_metrics' in locals() else 0.0,
    "ml_model_r2": step6_metrics.get("model_performance", {}).get("r2_score", 0.0) if 'step6_metrics' in locals() else 0.0,
    "all_artifacts_verified": True
}

print(f"   Total Inhibitors Screened: {project_metrics['summary']['total_inhibitors_screened']}")
print(f"   Novel Candidates Generated: {project_metrics['summary']['novel_candidates_generated']}")
print(f"   Final Leads Identified: {project_metrics['summary']['final_leads_identified']}")
print(f"   Best Predicted IC50: {project_metrics['summary']['best_predicted_ic50_nm']:.2f} nM")

# Save metrics to JSON
output_path = RESULTS_PATH / f"{TARGET_NAME}_project_summary_metrics.json"
print(f"\n" + "=" * 80)
print(f"SAVING CONSOLIDATED METRICS")
print(f"Output: {output_path}")
print("=" * 80)

# json_safe() turns NaN/Infinity into null and allow_nan=False makes a stray one
# an error rather than invalid output: json.dump otherwise writes a bare NaN, which
# Python reads back but JSON.parse, jq and most other parsers reject. All three of
# the earlier summary files were unparseable for exactly this reason.
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(pipeline_common.json_safe(project_metrics), f, indent=2, allow_nan=False)

print(f"\n✓ Successfully saved {output_path.name}")

# Verify all critical artifacts
print("\n" + "=" * 80)
print("VERIFYING CRITICAL ARTIFACTS")
print("=" * 80)

critical_files = [
    # Step 1
    f"results/{TARGET_NAME}_bioactivity_full.csv",
    f"results/{TARGET_NAME}_inhibitors_cleaned.csv",
    f"figures/{TARGET_NAME}_pic50_distribution.png",
    f"figures/{TARGET_NAME}_chemical_space_pca.png",
    f"figures/{TARGET_NAME}_physicochemical_properties.png",
    # Step 2
    f"results/{TARGET_NAME}_sar_analysis.csv",
    f"results/{TARGET_NAME}_scaffold_analysis.csv",
    f"figures/{TARGET_NAME}_top_scaffolds_potency.png",
    # Step 3
    f"results/{TARGET_NAME}_structural_analysis.json",
    f"results/{TARGET_NAME}_literature_findings.txt",
    # Step 4
    f"results/{TARGET_NAME}_selected_seeds.csv",
    f"results/{TARGET_NAME}_generated_candidates.csv",
    f"results/{TARGET_NAME}_top20_generated_candidates.csv",
    f"figures/{TARGET_NAME}_generation_pca.png",
    f"figures/{TARGET_NAME}_generation_tsne.png",
    # Step 5
    f"results/{TARGET_NAME}_docking_results.csv",
    f"results/{TARGET_NAME}_docking_site.json",
    f"figures/{TARGET_NAME}_docking_scores.png",
    # Step 6 -- these three used to be written without the target prefix, so a
    # second target in the same session silently overwrote them.
    f"results/{TARGET_NAME}_candidate_predictions.csv",
    f"results/{TARGET_NAME}_pIC50_model.pkl",
    f"results/{TARGET_NAME}_model_metrics.json",
    f"figures/{TARGET_NAME}_model_performance.png",
    f"figures/{TARGET_NAME}_ml_docking_comparison.png",
    # Step 7
    f"results/{TARGET_NAME}_admet_analysis.csv",
    f"results/{TARGET_NAME}_final_candidates.csv",
    f"figures/{TARGET_NAME}_candidate_radar_plot.png"
]

missing_files = []
for file_path in critical_files:
    full_path = BASE_PATH / file_path
    if full_path.exists():
        print(f"   ✓ {file_path}")
    else:
        print(f"   ✗ MISSING: {file_path}")
        missing_files.append(file_path)

project_metrics["summary"]["all_artifacts_verified"] = (len(missing_files) == 0)
project_metrics["summary"]["missing_artifacts"] = missing_files
_present = len(critical_files) - len(missing_files)
project_metrics["summary"]["pipeline_completeness"] = (
    f"{100.0 * _present / len(critical_files):.0f}%")
print(f"\n   Pipeline completeness: {_present}/{len(critical_files)} critical "
      f"artifacts present ({project_metrics['summary']['pipeline_completeness']})")
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(pipeline_common.json_safe(project_metrics), f, indent=2, allow_nan=False)

if missing_files:
    print(f"\n⚠️  Warning: {len(missing_files)} critical files are missing")
    print("   'all_artifacts_verified' recorded as false. It used to be hard-coded "
          "true while the\n   summary pointed at a model filename that did not exist.")
else:
    print(f"\n✓ All {len(critical_files)} critical artifacts verified")

print("\n" + "=" * 80)
print("CONSOLIDATION COMPLETE")
print("=" * 80)
print("\nProject is ready for handoff to writing agent.")
print(f"Summary metrics saved to: {output_path}")

In [ ]:
import os
from google.colab import files

# Name from the shared config, so the archive always matches the run.
import pipeline_common
CFG = pipeline_common.get_config(quiet=True)
project_name = CFG['target_name']

# Create a dynamic filename (e.g., "kras_project_backup.zip")
zip_filename = f"{project_name}_project_backup.zip"

print(f"📦 Compressing project files into: {zip_filename}...")

# 2. ZIP COMMAND
# We use f-string syntax in Python, so we pass the variable to the shell command using {zip_filename}
# Added quotes around "{zip_filename}" to handle spaces safely
!zip -r "{zip_filename}" . -x "sample_data/*" ".config/*" ".ipynb_checkpoints/*" "{zip_filename}"

# 3. CHECK SIZE & DOWNLOAD
file_size_mb = os.path.getsize(zip_filename) / 1024 / 1024
print(f"✓ Compression complete. Size: {file_size_mb:.2f} MB")

print(f"⬇️ Downloading {zip_filename}...")
files.download(zip_filename)